
# Projeto ED-ANTAQ-01
## Pipeline de Dados para Monitoramento da Movimentação Portuária

### Objetivo
Desenvolver um pipeline de Engenharia de Dados para leitura,
validação, transformação e armazenamento dos dados de
movimentação portuária da ANTAQ.

### Fonte
- Instituição: ANTAQ
- Base: Estatístico Aquaviário
- Ano inicial: 2024
- Arquivos:
  - 2024Carga.txt
  - 2024Atracacao.txt
  - 2024TemposAtracacao.txt

### Tecnologias
- Python
- Pandas
- Google Colab
- SQLite e PostgreSQL (etapas futuras)
- VS Code (etapa futura)

### Estratégia
1. Conhecer os dados brutos.
2. Investigar a estrutura das tabelas.
3. Identificar problemas de qualidade.
4. Desenvolver transformações reproduzíveis.
5. Armazenar os dados em banco SQL.
6. Automatizar e testar o pipeline.

### Escopo inicial
Exploração de uma amostra de 1.000 registros do arquivo
2024Carga.txt, sem modificar a fonte original.


In [1]:
import pandas as pd

print("Ambiente Python funcionando.")
print("Versão do Pandas:", pd.__version__)


Ambiente Python funcionando.
Versão do Pandas: 2.2.3


In [6]:
from pathlib import Path

pasta = Path("/content")

for arquivo in pasta.glob("2024*.txt"):
    tamanho_mb = arquivo.stat().st_size / (1024**2)

    print(f"Arquivo: {arquivo.name}")
    print(f"Tamanho: {tamanho_mb:.2f} MB")
    print("-" * 40)

Arquivo: 2024TemposAtracacao.txt
Tamanho: 10.57 MB
----------------------------------------
Arquivo: 2024Carga.txt
Tamanho: 430.24 MB
----------------------------------------
Arquivo: 2024Atracacao.txt
Tamanho: 35.12 MB
----------------------------------------


In [7]:
arquivo_carga = Path("/content/2024Carga.txt")

with open(arquivo_carga, "r", encoding="utf-8-sig") as f:
    for i in range(3):
        linha = f.readline()
        print(f"Linha {i+1}: {linha[:500]}")

Linha 1: IDCarga;IDAtracacao;Origem;Destino;CDMercadoria;Tipo Operação da Carga;Carga Geral Acondicionamento;ConteinerEstado;Tipo Navegação;FlagAutorizacao;FlagCabotagem;FlagCabotagemMovimentacao;FlagConteinerTamanho;FlagLongoCurso;FlagMCOperacaoCarga;FlagOffshore;FlagTransporteViaInterioir;Percurso Transporte em vias Interiores;Percurso Transporte Interiores;STNaturezaCarga;STSH2;STSH4;Natureza da Carga;Sentido;TEU;QTCarga;VLPesoCargaBruta

Linha 2: 35449975;1404798;BRVDC;BRVDC;CA01;Apoio;Solta;;Apoio Portuário;S;0;0;;0;1;0;0;;;Compartilhado;Compartilhado;Compartilhado;Carga Geral;Embarcados;0;90;3,98

Linha 3: 35449976;1404798;BRVDC;RULED;2201;Longo Curso Exportação;;;Longo Curso;S;0;0;;1;1;0;1;;Longo Curso em vias interiores;Compartilhado;Compartilhado;Compartilhado;Granel Líquido e Gasoso;Embarcados;0;0;17,66



In [8]:
import pandas as pd

arquivo_carga = "/content/2024Carga.txt"

df = pd.read_csv(
    arquivo_carga,
    sep=";",
    encoding="utf-8-sig",
    nrows=1000,
    low_memory=False
)

print("Quantidade de linhas:", df.shape[0])
print("Quantidade de colunas:", df.shape[1])

display(df.head())


Quantidade de linhas: 1000
Quantidade de colunas: 27


,IDCarga,IDAtracacao,Origem,Destino,CDMercadoria,Tipo Operação da Carga,Carga Geral Acondicionamento,ConteinerEstado,Tipo Navegação,FlagAutorizacao,...,Percurso Transporte em vias Interiores,Percurso Transporte Interiores,STNaturezaCarga,STSH2,STSH4,Natureza da Carga,Sentido,TEU,QTCarga,VLPesoCargaBruta
0,35449975,1404798,BRVDC,BRVDC,CA01,Apoio,Solta,NaN,Apoio Portuário,S,...,NaN,NaN,Compartilhado,Compartilhado,Compartilhado,Carga Geral,Embarcados,0,90,"3,98"
1,35449976,1404798,BRVDC,RULED,2201,Longo Curso Exportação,NaN,NaN,Longo Curso,S,...,NaN,Longo Curso em vias interiores,Compartilhado,Compartilhado,Compartilhado,Granel Líquido e Gasoso,Embarcados,0,0,"17,66"
2,35449977,1404798,BRVDC,CAPAF,2818,Longo Curso Exportação,NaN,NaN,Longo Curso,S,...,NaN,Longo Curso em vias interiores,Compartilhado,Compartilhado,Compartilhado,Granel Sólido,Embarcados,0,0,"34452,28"
3,35449978,1404801,BRES002,BRIGI,2601,Cabotagem,NaN,NaN,Cabotagem,S,...,NaN,NaN,Exclusivo,Exclusivo,Exclusivo,Granel Sólido,Embarcados,0,0,60499
4,35450024,1404849,USMOB,BRRJ016,2701,Longo Curso Importação,NaN,NaN,Longo Curso,S,...,NaN,NaN,Exclusivo,Exclusivo,Exclusivo,Granel Sólido,Desembarcados,0,0,72273


In [10]:
# Verificar o tipo de cada coluna

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 27 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   IDCarga                                 1000 non-null   int64  
 1   IDAtracacao                             1000 non-null   int64  
 2   Origem                                  999 non-null    object 
 3   Destino                                 999 non-null    object 
 4   CDMercadoria                            1000 non-null   object 
 5   Tipo Operação da Carga                  1000 non-null   object 
 6   Carga Geral Acondicionamento            786 non-null    object 
 7   ConteinerEstado                         2 non-null      object 
 8   Tipo Navegação                          1000 non-null   object 
 9   FlagAutorizacao                         1000 non-null   object 
 10  FlagCabotagem                           1000 non-null   int64

In [11]:
# Criar uma cópia da amostra original
df_tratado = df.copy()

# Converter o peso de texto para número

df_tratado["VLPesoCargaBruta"] = pd.to_numeric(
    df_tratado["VLPesoCargaBruta"].str.replace(",", ".", regex=False),
    errors="coerce"
)

print("Tipo original:", df["VLPesoCargaBruta"].dtype)
print("Tipo tratado:", df_tratado["VLPesoCargaBruta"].dtype)

display(
    pd.DataFrame({
        "Peso original": df["VLPesoCargaBruta"].head(10),
        "Peso convertido": df_tratado["VLPesoCargaBruta"].head(10)
    })
)

Tipo original: object
Tipo tratado: float64


,Peso original,Peso convertido
0,"3,98",3.98
1,"17,66",17.66
2,"34452,28",34452.28
3,60499,60499.00
4,72273,72273.00
5,"1,92",1.92
6,"4,68",4.68
7,"0,68",0.68
8,"161,04",161.04
9,"2964,12",2964.12



### Registro de transformação: VLPesoCargaBruta

**Problema identificado:**
O campo de peso foi reconhecido pelo Pandas como `object`,
pois os valores da amostra utilizam vírgula decimal.

**Tratamento aplicado:**
Conversão para formato numérico, utilizando ponto decimal
na representação interna do Python.

**Cuidados:**
- Preservar os valores originais.
- Não excluir registros automaticamente.
- Identificar falhas de conversão.
- Validar a premissa de formatação na base completa.

**Resultado na amostra de 1.000 registros:**
- Nulos antes: 0
- Nulos depois: 0
- Falhas de conversão: 0

**Status:**
Validação inicial aprovada para a amostra.
Pendente de validação na base completa.


In [12]:
# Quantidade de valores ausentes antes
nulos_antes = df["VLPesoCargaBruta"].isna().sum()

# Quantidade de valores ausentes depois
nulos_depois = df_tratado["VLPesoCargaBruta"].isna().sum()

# Valores que estavam preenchidos, mas falharam na conversão
falhas_conversao = (
    df["VLPesoCargaBruta"].notna()
    & df_tratado["VLPesoCargaBruta"].isna()
)

print("Nulos antes:", nulos_antes)
print("Nulos depois:", nulos_depois)
print("Falhas de conversão:", falhas_conversao.sum())

Nulos antes: 0
Nulos depois: 0
Falhas de conversão: 0



## Aula 2 — Chaves e integridade dos dados

### Objetivo
Investigar as chaves candidatas da tabela de cargas e verificar
a existência de identificadores nulos ou repetidos.

### Hipóteses iniciais
- IDCarga pode identificar exclusivamente cada registro de carga.
- IDAtracacao pode relacionar cargas e atracações.
- Uma atracação pode possuir múltiplos registros de carga.

As hipóteses serão verificadas inicialmente em uma amostra
de 1.000 registros.


In [13]:
# AULA 2 - TESTE DE UNICIDADE DA CHAVE IDCarga

total_registros = len(df_tratado)

ids_unicos = df_tratado["IDCarga"].nunique()

ids_nulos = df_tratado["IDCarga"].isna().sum()

ids_duplicados = df_tratado["IDCarga"].duplicated().sum()

print("Total de registros:", total_registros)
print("IDs únicos:", ids_unicos)
print("IDs nulos:", ids_nulos)
print("IDs duplicados:", ids_duplicados)


Total de registros: 1000
IDs únicos: 1000
IDs nulos: 0
IDs duplicados: 0


In [14]:
# AULA 2 - INVESTIGAR O RELACIONAMENTO COM ATRACAÇÕES

atracacoes_unicas = df_tratado["IDAtracacao"].nunique()

atracacoes_nulas = df_tratado["IDAtracacao"].isna().sum()

registros_por_atracacao = (
    df_tratado
    .groupby("IDAtracacao")
    .size()
    .sort_values(ascending=False)
)

print("Atracações distintas:", atracacoes_unicas)
print("Referências nulas:", atracacoes_nulas)

print("\n10 atracações com mais registros de carga:")
display(registros_por_atracacao.head(10))


Atracações distintas: 164
Referências nulas: 0

10 atracações com mais registros de carga:


,0
IDAtracacao,
1409040,55
1409041,49
1408839,48
1408841,45
1409038,44
1408845,44
1408837,42
1409039,40
1408846,37


In [15]:
# AULA 2 - DUPLICIDADE DE LINHAS COMPLETAS

linhas_duplicadas = df_tratado.duplicated().sum()

print("Linhas inteiramente duplicadas:", linhas_duplicadas)

# Exibir registros repetidos, caso existam
if linhas_duplicadas > 0:
    display(
        df_tratado[
            df_tratado.duplicated(keep=False)
        ].sort_values("IDCarga")
    )


Linhas inteiramente duplicadas: 0



### Resultados da validação de chaves

**Tabela:** 2024Carga.txt

**Amostra:** 1.000 registros

#### IDCarga
- Quantidade de IDs distintos:
- Quantidade de IDs nulos:
- Quantidade de ocorrências duplicadas:

#### IDAtracacao
- Quantidade de atracações distintas:
- Quantidade de referências nulas:
- Maior quantidade de registros por atracação:

#### Duplicidades completas
- Quantidade identificada:

### Conclusão
A preencher após a execução dos testes.

### Limitações
Resultados válidos somente para a amostra analisada.
A integridade referencial com o arquivo 2024Atracacao.txt
ainda não foi verificada.



# Aula 3 — Integração de Carga e Atracação **negrito**

### Objetivo
Relacionar registros de carga às respectivas atracações
utilizando o campo IDAtracacao.

### Fontes
- 2024Carga.txt
- 2024Atracacao.txt

### Hipótese de relacionamento
Uma atracação pode estar associada a múltiplos registros
de carga (relação 1:N).

### Regras de qualidade
1. Verificar se IDAtracacao é único em Atracacao.
2. Identificar cargas sem atracação correspondente.
3. Garantir que o relacionamento não multiplique linhas.
4. Preservar os valores originais de peso.

### Limitação
Nesta aula utilizaremos uma amostra de cargas.
A validação do arquivo completo será realizada posteriormente.


In [16]:
from pathlib import Path

arquivo_atracacao = Path("/content/2024Atracacao.txt")

with open(
    arquivo_atracacao,
    "r",
    encoding="utf-8-sig"
) as arquivo:
    for i in range(3):
        linha = arquivo.readline()
        print(f"Linha {i + 1}: {linha[:500]}")


Linha 1: IDAtracacao;CDTUP;IDBerco;Berço;Porto Atracação;Coordenadas;Apelido Instalação Portuária;Complexo Portuário;Tipo da Autoridade Portuária;Data Atracação;Data Chegada;Data Desatracação;Data Início Operação;Data Término Operação;Ano;Mes;Tipo de Operação;Tipo de Navegação da Atracação;Nacionalidade do Armador;FlagMCOperacaoAtracacao;Terminal;Município;UF;SGUF;Região Geográfica;Região Hidrográfica;Instalação Portuária em Rio;Nº da Capitania;Nº do IMO

Linha 2: 1512136;BRNAT;NAT0103;B3;Natal;-35.204999,-5.772777;;Natal;Porto Organizado;02/11/2024 09:10:00;02/11/2024 09:00:00;06/11/2024 02:00:00;;;2024;nov;Marinha;Cabotagem;0;1;Cais Comercial;Natal;Rio Grande do Norte;RN;Nordeste;;Não;;

Linha 3: 1466554;BRIBB;IBB0002;Berço 2;Imbituba;-48.65416700000001,-28.231389;;Imbituba;Porto Organizado;06/06/2024 14:30:00;06/06/2024 09:00:00;14/07/2024 19:10:00;;;2024;jul;Marinha;Apoio Marítimo;0;1;Cais Público;Imbituba;Santa Catarina;SC;Sul;;Não;;



In [17]:

import pandas as pd

df_atracacao = pd.read_csv(
    "/content/2024Atracacao.txt",
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

print("Linhas:", df_atracacao.shape[0])
print("Colunas:", df_atracacao.shape[1])

display(df_atracacao.head())

print("\nNomes das colunas:")
print(df_atracacao.columns.tolist())


Linhas: 106834
Colunas: 29


,IDAtracacao,CDTUP,IDBerco,Berço,Porto Atracação,Coordenadas,Apelido Instalação Portuária,Complexo Portuário,Tipo da Autoridade Portuária,Data Atracação,...,FlagMCOperacaoAtracacao,Terminal,Município,UF,SGUF,Região Geográfica,Região Hidrográfica,Instalação Portuária em Rio,Nº da Capitania,Nº do IMO
0,1512136,BRNAT,NAT0103,B3,Natal,"-35.204999,-5.772777",NaN,Natal,Porto Organizado,02/11/2024 09:10:00,...,1,Cais Comercial,Natal,Rio Grande do Norte,RN,Nordeste,NaN,Não,NaN,NaN
1,1466554,BRIBB,IBB0002,Berço 2,Imbituba,"-48.65416700000001,-28.231389",NaN,Imbituba,Porto Organizado,06/06/2024 14:30:00,...,1,Cais Público,Imbituba,Santa Catarina,SC,Sul,NaN,Não,NaN,NaN
2,1484269,BRIBB,IBB0002,Berço 2,Imbituba,"-48.65416700000001,-28.231389",NaN,Imbituba,Porto Organizado,11/08/2024 17:15:00,...,1,Cais Público,Imbituba,Santa Catarina,SC,Sul,NaN,Não,NaN,NaN
3,1433333,BRMCZ,MCZ0002,Berço 2,Maceió,"-35.726388,-9.683056000000001",NaN,Maceió,Porto Organizado,15/03/2024 09:00:00,...,1,Cais do AA-01,Maceió,Alagoas,AL,Nordeste,NaN,Não,NaN,NaN
4,1489223,BRFOR,FOR0105,Berço 105,Fortaleza,"-38.47361,-3.709722000000001",Mucuripe,Pecém - Fortaleza,Porto Organizado,06/08/2024 08:49:00,...,1,Cais Comercial,Fortaleza,Ceará,CE,Nordeste,NaN,Não,NaN,NaN



Nomes das colunas:
['IDAtracacao', 'CDTUP', 'IDBerco', 'Berço', 'Porto Atracação', 'Coordenadas', 'Apelido Instalação Portuária', 'Complexo Portuário', 'Tipo da Autoridade Portuária', 'Data Atracação', 'Data Chegada', 'Data Desatracação', 'Data Início Operação', 'Data Término Operação', 'Ano', 'Mes', 'Tipo de Operação', 'Tipo de Navegação da Atracação', 'Nacionalidade do Armador', 'FlagMCOperacaoAtracacao', 'Terminal', 'Município', 'UF', 'SGUF', 'Região Geográfica', 'Região Hidrográfica', 'Instalação Portuária em Rio', 'Nº da Capitania', 'Nº do IMO']


In [18]:
import pandas as pd

df_atracacao = pd.read_csv(
    "/content/2024Atracacao.txt",
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

print("Linhas:", df_atracacao.shape[0])
print("Colunas:", df_atracacao.shape[1])

display(df_atracacao.head())

print("\nNomes das colunas:")
print(df_atracacao.columns.tolist())


Linhas: 106834
Colunas: 29


,IDAtracacao,CDTUP,IDBerco,Berço,Porto Atracação,Coordenadas,Apelido Instalação Portuária,Complexo Portuário,Tipo da Autoridade Portuária,Data Atracação,...,FlagMCOperacaoAtracacao,Terminal,Município,UF,SGUF,Região Geográfica,Região Hidrográfica,Instalação Portuária em Rio,Nº da Capitania,Nº do IMO
0,1512136,BRNAT,NAT0103,B3,Natal,"-35.204999,-5.772777",NaN,Natal,Porto Organizado,02/11/2024 09:10:00,...,1,Cais Comercial,Natal,Rio Grande do Norte,RN,Nordeste,NaN,Não,NaN,NaN
1,1466554,BRIBB,IBB0002,Berço 2,Imbituba,"-48.65416700000001,-28.231389",NaN,Imbituba,Porto Organizado,06/06/2024 14:30:00,...,1,Cais Público,Imbituba,Santa Catarina,SC,Sul,NaN,Não,NaN,NaN
2,1484269,BRIBB,IBB0002,Berço 2,Imbituba,"-48.65416700000001,-28.231389",NaN,Imbituba,Porto Organizado,11/08/2024 17:15:00,...,1,Cais Público,Imbituba,Santa Catarina,SC,Sul,NaN,Não,NaN,NaN
3,1433333,BRMCZ,MCZ0002,Berço 2,Maceió,"-35.726388,-9.683056000000001",NaN,Maceió,Porto Organizado,15/03/2024 09:00:00,...,1,Cais do AA-01,Maceió,Alagoas,AL,Nordeste,NaN,Não,NaN,NaN
4,1489223,BRFOR,FOR0105,Berço 105,Fortaleza,"-38.47361,-3.709722000000001",Mucuripe,Pecém - Fortaleza,Porto Organizado,06/08/2024 08:49:00,...,1,Cais Comercial,Fortaleza,Ceará,CE,Nordeste,NaN,Não,NaN,NaN



Nomes das colunas:
['IDAtracacao', 'CDTUP', 'IDBerco', 'Berço', 'Porto Atracação', 'Coordenadas', 'Apelido Instalação Portuária', 'Complexo Portuário', 'Tipo da Autoridade Portuária', 'Data Atracação', 'Data Chegada', 'Data Desatracação', 'Data Início Operação', 'Data Término Operação', 'Ano', 'Mes', 'Tipo de Operação', 'Tipo de Navegação da Atracação', 'Nacionalidade do Armador', 'FlagMCOperacaoAtracacao', 'Terminal', 'Município', 'UF', 'SGUF', 'Região Geográfica', 'Região Hidrográfica', 'Instalação Portuária em Rio', 'Nº da Capitania', 'Nº do IMO']


In [19]:

# Verificar se cada carga encontra uma atracação

ids_atracacao = set(
    df_atracacao["IDAtracacao"].dropna()
)

df_validacao = df_tratado[
    ["IDCarga", "IDAtracacao"]
].copy()

df_validacao["AtracacaoEncontrada"] = (
    df_validacao["IDAtracacao"]
    .isin(ids_atracacao)
)

print("Cargas analisadas:", len(df_validacao))

print(
    "\nSituação do relacionamento:"
)

print(
    df_validacao["AtracacaoEncontrada"]
    .value_counts(dropna=False)
)


Cargas analisadas: 1000

Situação do relacionamento:
AtracacaoEncontrada
True    1000
Name: count, dtype: int64


In [20]:

# Selecionar algumas informações da tabela de atracações

colunas_atracacao = [
    "IDAtracacao",
    "Porto Atracação",
    "Data Atracação"
]

# Manter apenas as colunas realmente existentes
colunas_disponiveis = [
    coluna
    for coluna in colunas_atracacao
    if coluna in df_atracacao.columns
]

df_atracacao_selecionado = (
    df_atracacao[colunas_disponiveis].copy()
)

# Relacionamento da carga com a atracação
df_integrado = df_tratado.merge(
    df_atracacao_selecionado,
    on="IDAtracacao",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Linhas antes:", len(df_tratado))
print("Linhas depois:", len(df_integrado))

print("\nResultado do relacionamento:")
print(df_integrado["_merge"].value_counts())

display(df_integrado.head())


Linhas antes: 1000
Linhas depois: 1000

Resultado do relacionamento:
_merge
both          1000
left_only        0
right_only       0
Name: count, dtype: int64


,IDCarga,IDAtracacao,Origem,Destino,CDMercadoria,Tipo Operação da Carga,Carga Geral Acondicionamento,ConteinerEstado,Tipo Navegação,FlagAutorizacao,...,STSH2,STSH4,Natureza da Carga,Sentido,TEU,QTCarga,VLPesoCargaBruta,Porto Atracação,Data Atracação,_merge
0,35449975,1404798,BRVDC,BRVDC,CA01,Apoio,Solta,NaN,Apoio Portuário,S,...,Compartilhado,Compartilhado,Carga Geral,Embarcados,0,90,3.98,Vila do Conde,28/12/2023 11:46:00,both
1,35449976,1404798,BRVDC,RULED,2201,Longo Curso Exportação,NaN,NaN,Longo Curso,S,...,Compartilhado,Compartilhado,Granel Líquido e Gasoso,Embarcados,0,0,17.66,Vila do Conde,28/12/2023 11:46:00,both
2,35449977,1404798,BRVDC,CAPAF,2818,Longo Curso Exportação,NaN,NaN,Longo Curso,S,...,Compartilhado,Compartilhado,Granel Sólido,Embarcados,0,0,34452.28,Vila do Conde,28/12/2023 11:46:00,both
3,35449978,1404801,BRES002,BRIGI,2601,Cabotagem,NaN,NaN,Cabotagem,S,...,Exclusivo,Exclusivo,Granel Sólido,Embarcados,0,0,60499.00,Terminal Marítimo Ponta Ubu,30/12/2023 23:50:00,both
4,35450024,1404849,USMOB,BRRJ016,2701,Longo Curso Importação,NaN,NaN,Longo Curso,S,...,Exclusivo,Exclusivo,Granel Sólido,Desembarcados,0,0,72273.00,Terminal TKCSA,27/12/2023 13:25:00,both


In [21]:

# Verificar se a quantidade de linhas foi preservada
assert len(df_integrado) == len(df_tratado), (
    "Atenção: a quantidade de linhas foi alterada!"
)

# Comparar o peso antes e depois
peso_antes = df_tratado["VLPesoCargaBruta"].sum()

peso_depois = df_integrado["VLPesoCargaBruta"].sum()

print("Peso antes:", peso_antes)
print("Peso depois:", peso_depois)

print("Diferença:", peso_depois - peso_antes)


Peso antes: 3289718.485
Peso depois: 3289718.485
Diferença: 0.0



# **## Aula 4 — Integração das bases ANTAQ**

### Objetivo
Relacionar registros da tabela Carga aos registros da
tabela Atracacao utilizando IDAtracacao.

### Regra de relacionamento
- Tabela principal: Carga
- Tabela complementar: Atracacao
- Tipo de relacionamento: LEFT JOIN
- Cardinalidade esperada: muitos para um (N:1)

### Validações
1. A chave deve ser única na tabela Atracacao.
2. A quantidade de registros de carga deve ser preservada.
3. O peso total deve permanecer consistente.
4. Cargas sem correspondência devem ser identificadas.

### Limitação
A base de carga ainda corresponde a uma amostra de
1.000 registros.


In [22]:
# AULA 4 - RELACIONAMENTO ENTRE CARGA E ATRACAÇÃO

# Colunas disponíveis na tabela de atracações
print("Colunas da tabela Atracacao:")
print(df_atracacao.columns.tolist())

# Selecionar a chave e até quatro campos adicionais
colunas_extras = [
    coluna
    for coluna in df_atracacao.columns
    if coluna != "IDAtracacao"
][:4]

colunas_selecionadas = [
    "IDAtracacao",
    *colunas_extras
]

df_atracacao_aux = df_atracacao[
    colunas_selecionadas
].copy()

# Validar a chave da tabela complementar
assert df_atracacao_aux["IDAtracacao"].notna().all()
assert df_atracacao_aux["IDAtracacao"].is_unique

# Relacionar as tabelas
df_integrado = df_tratado.merge(
    df_atracacao_aux,
    on="IDAtracacao",
    how="left",
    validate="many_to_one",
    indicator=True,
    suffixes=("", "_atracacao")
)

print("Linhas de carga:", len(df_tratado))
print("Linhas após JOIN:", len(df_integrado))

print("\nCorrespondências:")
print(df_integrado["_merge"].value_counts())

display(df_integrado.head())


Colunas da tabela Atracacao:
['IDAtracacao', 'CDTUP', 'IDBerco', 'Berço', 'Porto Atracação', 'Coordenadas', 'Apelido Instalação Portuária', 'Complexo Portuário', 'Tipo da Autoridade Portuária', 'Data Atracação', 'Data Chegada', 'Data Desatracação', 'Data Início Operação', 'Data Término Operação', 'Ano', 'Mes', 'Tipo de Operação', 'Tipo de Navegação da Atracação', 'Nacionalidade do Armador', 'FlagMCOperacaoAtracacao', 'Terminal', 'Município', 'UF', 'SGUF', 'Região Geográfica', 'Região Hidrográfica', 'Instalação Portuária em Rio', 'Nº da Capitania', 'Nº do IMO']
Linhas de carga: 1000
Linhas após JOIN: 1000

Correspondências:
_merge
both          1000
left_only        0
right_only       0
Name: count, dtype: int64


,IDCarga,IDAtracacao,Origem,Destino,CDMercadoria,Tipo Operação da Carga,Carga Geral Acondicionamento,ConteinerEstado,Tipo Navegação,FlagAutorizacao,...,Natureza da Carga,Sentido,TEU,QTCarga,VLPesoCargaBruta,CDTUP,IDBerco,Berço,Porto Atracação,_merge
0,35449975,1404798,BRVDC,BRVDC,CA01,Apoio,Solta,NaN,Apoio Portuário,S,...,Carga Geral,Embarcados,0,90,3.98,BRVDC,VDC0201,Berço 201,Vila do Conde,both
1,35449976,1404798,BRVDC,RULED,2201,Longo Curso Exportação,NaN,NaN,Longo Curso,S,...,Granel Líquido e Gasoso,Embarcados,0,0,17.66,BRVDC,VDC0201,Berço 201,Vila do Conde,both
2,35449977,1404798,BRVDC,CAPAF,2818,Longo Curso Exportação,NaN,NaN,Longo Curso,S,...,Granel Sólido,Embarcados,0,0,34452.28,BRVDC,VDC0201,Berço 201,Vila do Conde,both
3,35449978,1404801,BRES002,BRIGI,2601,Cabotagem,NaN,NaN,Cabotagem,S,...,Granel Sólido,Embarcados,0,0,60499.00,BRES002,BRES0020001,BERÇO OESTE,Terminal Marítimo Ponta Ubu,both
4,35450024,1404849,USMOB,BRRJ016,2701,Longo Curso Importação,NaN,NaN,Longo Curso,S,...,Granel Sólido,Desembarcados,0,0,72273.00,BRRJ016,BRRJ0160001,Berço 1,Terminal TKCSA,both


In [23]:
# AULA 4 - VALIDAÇÃO DA INTEGRAÇÃO

# Verificação 1: preservar quantidade de registros
assert len(df_integrado) == len(df_tratado), (
    "ERRO: quantidade de linhas alterada!"
)

# Verificação 2: preservar cada registro de carga
assert df_integrado["IDCarga"].equals(
    df_tratado["IDCarga"]
), "ERRO: IDs de carga foram alterados!"

# Verificação 3: preservar os pesos originais
import numpy as np

assert np.allclose(
    df_integrado["VLPesoCargaBruta"].to_numpy(),
    df_tratado["VLPesoCargaBruta"].to_numpy(),
    equal_nan=True,
    rtol=0,
    atol=0
), "ERRO: valores de peso alterados!"

# Verificação 4: identificar cargas sem atracação
nao_encontradas = (
    df_integrado["_merge"] == "left_only"
).sum()

print("Validação de linhas: OK")
print("Validação de IDs: OK")
print("Validação de pesos: OK")
print("Cargas sem correspondência:", nao_encontradas)


Validação de linhas: OK
Validação de IDs: OK
Validação de pesos: OK
Cargas sem correspondência: 0


In [24]:
# Procurar possíveis colunas relacionadas a portos
palavras_chave = [
    "porto",
    "instal",
    "complexo",
    "terminal",
    "local"
]

colunas_localizacao = [
    coluna
    for coluna in df_atracacao.columns
    if any(
        palavra in coluna.lower()
        for palavra in palavras_chave
    )
]

print("Possíveis colunas de localização:")
for coluna in colunas_localizacao:
    print("-", coluna)


Possíveis colunas de localização:
- Porto Atracação
- Apelido Instalação Portuária
- Complexo Portuário
- Terminal
- Instalação Portuária em Rio



# Resultado da integração **negrito**

**Tabela principal:** Carga  
**Tabela complementar:** Atracacao  
**Chave:** IDAtracacao  
**Relacionamento:** LEFT JOIN (N:1)

#### Indicadores
- Registros antes:
- Registros depois:
- Registros com correspondência:
- Registros sem correspondência:
- Validação dos pesos:

#### Conclusão
Descrever se a amostra passou nas verificações.

#### Limitações
- Carga analisada em amostra.
- Validação nacional completa ainda pendente.
- Recorte do Porto de Santos ainda não aplicado.


In [25]:
print("1. RESULTADO DO RELACIONAMENTO")
print(df_integrado["_merge"].value_counts())

print("\n2. QUANTIDADE DE REGISTROS")
print("Antes:", len(df_tratado))
print("Depois:", len(df_integrado))

print("\n3. COLUNAS DE LOCALIZAÇÃO")
print(colunas_localizacao)

1. RESULTADO DO RELACIONAMENTO
_merge
both          1000
left_only        0
right_only       0
Name: count, dtype: int64

2. QUANTIDADE DE REGISTROS
Antes: 1000
Depois: 1000

3. COLUNAS DE LOCALIZAÇÃO
['Porto Atracação', 'Apelido Instalação Portuária', 'Complexo Portuário', 'Terminal', 'Instalação Portuária em Rio']



## Aula 5 — Identificação do Porto de Santos

### Objetivo
Identificar o campo de localização adequado para selecionar
as movimentações de carga associadas ao Porto de Santos.

### Campos investigados
- Porto Atracação
- Apelido Instalação Portuária
- Complexo Portuário
- Terminal
- Instalação Portuária em Rio

### Decisões metodológicas
1. Investigar os valores antes de aplicar filtros.
2. Distinguir porto de complexo portuário.
3. Utilizar uma classificação exata após validação.
4. Não considerar a amostra de 1.000 cargas representativa
   da movimentação nacional.

### Status
Em investigação. Filtro definitivo ainda não aplicado.


In [26]:
# AULA 5 - INVESTIGAR O CAMPO PORTO ATRACAÇÃO

print("Portos encontrados na amostra:")

print(
    df_atracacao["Porto Atracação"]
    .value_counts(dropna=False)
    .head(20)
)


Portos encontrados na amostra:
Porto Atracação
Brasil Logística Offshore e Estaleiro Naval          9522
Belém                                                6275
Santos                                               4960
Santarém                                             4291
Rio de Janeiro                                       2927
Terminal Graneleiro Hermasa                          2634
Rio Grande                                           2609
Paranaguá                                            2353
Terminal Vila do Conde                               2287
Vila do Conde                                        2256
Porto de Guamaré                                     2219
Hidrovias do Brasil Miritituba                       2182
Suape                                                1807
Porto Gregório Curvo                                 1721
TUP J F de Oliveira de Belém                         1712
ETC Miritituba                                       1673
Terminal Portuário Granel

In [27]:
# Identificar valores que contêm a palavra Santos

portos_santos = (
    df_atracacao.loc[
        df_atracacao["Porto Atracação"]
        .astype("string")
        .str.contains("Santos", case=False, na=False),
        "Porto Atracação"
    ]
    .value_counts(dropna=False)
)

print(portos_santos)


Porto Atracação
Santos             4960
DP World Santos     727
Name: count, dtype: int64


In [28]:
# Localizar colunas de localização disponíveis na base integrada
colunas_encontradas = [
    coluna
    for coluna in df_integrado.columns
    if "Porto" in coluna
    or "porto" in coluna
]

print("Colunas relacionadas a porto:")
print(colunas_encontradas)

# Verificar se Porto Atracação foi incorporado
if "Porto Atracação" in df_integrado.columns:

    print("\nPortos presentes na amostra de carga:")
    print(
        df_integrado["Porto Atracação"]
        .value_counts(dropna=False)
        .head(20)
    )

else:
    print(
        "\nO campo Porto Atracação não foi incluído "
        "na seleção de colunas do JOIN anterior."
    )


Colunas relacionadas a porto:
['Porto Atracação']

Portos presentes na amostra de carga:
Porto Atracação
TUP J F de Oliveira de Belém                                          436
ATR Logística - Chibatão                                              318
Terminal Integrador Portuário Luiz Antonio Mesquita - TIPLAM           60
Vila do Conde                                                          46
Terminais Fluviais do Brasil                                           28
Belém                                                                  20
Terminal Portuário Cotegipe                                            11
Terminal de Expedição de Grãos Portochuelo                              9
Terminal Aquaviário de Angra dos Reis                                   9
ABI Miritituba                                                          6
Terminal Trombetas                                                      6
Terminal Navecunha                                                      5
Companh

In [29]:

# AULA 5 - INVESTIGAR CLASSIFICAÇÕES PORTUÁRIAS

colunas_analise = [
    "Porto Atracação",
    "Apelido Instalação Portuária",
    "Complexo Portuário"
]

# Selecionar Santos e DP World Santos
df_portos = df_atracacao[
    df_atracacao["Porto Atracação"].isin(
        ["Santos", "DP World Santos"]
    )
].copy()

# Exibir combinações distintas
resumo_portos = (
    df_portos[colunas_analise]
    .value_counts(dropna=False)
    .reset_index(name="QtdAtracacoes")
    .sort_values(
        "QtdAtracacoes",
        ascending=False
    )
)

display(resumo_portos)


,Porto Atracação,Apelido Instalação Portuária,Complexo Portuário,QtdAtracacoes
0,Santos,NaN,Santos,4960
1,DP World Santos,Ex-Embraport,Santos,727



### Investigação do recorte Porto de Santos

**Fonte:** ANTAQ - 2024Atracacao.txt

#### Resultados iniciais
- Porto Atracação = Santos: 4.960 registros
- Porto Atracação = DP World Santos: 727 registros

#### Observação
A ANTAQ apresenta Santos e DP World Santos
como valores distintos no campo Porto Atracação.

#### Decisão metodológica
Ainda não consolidar os registros.

Verificar as classificações de instalação e complexo
portuário antes de definir o recorte analítico.

#### Próxima etapa
Construir uma regra de classificação documentada
e selecionar os registros correspondentes.


In [30]:
# AULA 6 - TABELA DE REFERÊNCIA DO COMPLEXO DE SANTOS

colunas_referencia = [
    "IDAtracacao",
    "Porto Atracação",
    "Apelido Instalação Portuária",
    "Complexo Portuário"
]

# Selecionar atracações do Complexo Portuário de Santos
df_santos_atracacoes = df_atracacao.loc[
    df_atracacao["Complexo Portuário"] == "Santos",
    colunas_referencia
].copy()

# Validar integridade da chave
assert df_santos_atracacoes["IDAtracacao"].notna().all()
assert df_santos_atracacoes["IDAtracacao"].is_unique

print("Atracações do complexo:",
      len(df_santos_atracacoes))

print("\nDistribuição por porto:")
print(
    df_santos_atracacoes["Porto Atracação"]
    .value_counts(dropna=False)
)


Atracações do complexo: 6292

Distribuição por porto:
Porto Atracação
Santos                                                          4960
DP World Santos                                                  727
Terminal Integrador Portuário Luiz Antonio Mesquita - TIPLAM     332
Sucocítrico Cutrale                                               90
Base Logística de Dutos                                           77
Terminal Marítimo Dow                                             63
Terminal Marítimo Privativo de Cubatão - TMPC                     43
Name: count, dtype: int64


In [31]:

# AULA 6 - LEITURA EM BLOCOS

arquivo_carga = "/content/2024Carga.txt"

# IDs associados ao Complexo Portuário de Santos
ids_santos = set(
    df_santos_atracacoes["IDAtracacao"]
)

tamanho_bloco = 50000

# Variáveis de controle
total_lidos = 0
total_santos = 0
blocos_santos = []

leitor = pd.read_csv(
    arquivo_carga,
    sep=";",
    encoding="utf-8-sig",
    chunksize=tamanho_bloco,
    low_memory=False
)

# Processar apenas os dois primeiros blocos
for numero_bloco, bloco in enumerate(leitor, start=1):

    if numero_bloco > 2:
        break

    total_lidos += len(bloco)

    # Selecionar cargas associadas a Santos
    filtro = bloco["IDAtracacao"].isin(ids_santos)

    bloco_santos = bloco.loc[filtro].copy()

    total_santos += len(bloco_santos)

    blocos_santos.append(bloco_santos)

    print(
        f"Bloco {numero_bloco}: "
        f"{len(bloco):,} registros lidos | "
        f"{len(bloco_santos):,} registros de Santos"
    )

print("\nRESUMO")
print("Registros nacionais lidos:", total_lidos)
print("Registros selecionados:", total_santos)


Bloco 1: 50,000 registros lidos | 35,532 registros de Santos
Bloco 2: 50,000 registros lidos | 16,103 registros de Santos

RESUMO
Registros nacionais lidos: 100000
Registros selecionados: 51635


In [32]:

# Consolidar os registros selecionados no teste

if blocos_santos:
    df_carga_santos_teste = pd.concat(
        blocos_santos,
        ignore_index=True
    )
else:
    df_carga_santos_teste = pd.DataFrame(
        columns=df.columns
    )

print("Linhas selecionadas:",
      len(df_carga_santos_teste))

print("Colunas:",
      len(df_carga_santos_teste.columns))

display(df_carga_santos_teste.head())


Linhas selecionadas: 51635
Colunas: 27


,IDCarga,IDAtracacao,Origem,Destino,CDMercadoria,Tipo Operação da Carga,Carga Geral Acondicionamento,ConteinerEstado,Tipo Navegação,FlagAutorizacao,...,Percurso Transporte em vias Interiores,Percurso Transporte Interiores,STNaturezaCarga,STSH2,STSH4,Natureza da Carga,Sentido,TEU,QTCarga,VLPesoCargaBruta
0,35450474,1405071,BRSP001,CLPMC,2304,Longo Curso Exportação,NaN,NaN,Longo Curso,S,...,NaN,NaN,Exclusivo,Exclusivo,Exclusivo,Granel Sólido,Embarcados,0,0,26300
1,35457571,1406996,BRARB,BRSP005,2910,Cabotagem,NaN,NaN,Cabotagem,S,...,NaN,NaN,Exclusivo,Compartilhado,Compartilhado,Granel Líquido e Gasoso,Desembarcados,0,0,2443
2,35457572,1406996,BRARB,BRSP005,2905,Cabotagem,NaN,NaN,Cabotagem,S,...,NaN,NaN,Exclusivo,Compartilhado,Compartilhado,Granel Líquido e Gasoso,Desembarcados,0,0,907
3,35457573,1406996,BRARB,BRSP005,2909,Cabotagem,NaN,NaN,Cabotagem,S,...,NaN,NaN,Exclusivo,Compartilhado,Compartilhado,Granel Líquido e Gasoso,Desembarcados,0,0,150
4,35457574,1406996,BRSP005,BRARB,3907,Cabotagem,NaN,NaN,Cabotagem,S,...,NaN,NaN,Exclusivo,Compartilhado,Compartilhado,Granel Líquido e Gasoso,Embarcados,0,0,350



## Aula 6 — Processamento em blocos

### Objetivo
Selecionar registros de movimentação de cargas vinculados
ao Complexo Portuário de Santos.

### Fonte
ANTAQ — Estatístico Aquaviário, ano 2024.

### Regra territorial
Complexo Portuário = Santos.

### Tabela de referência
2024Atracacao.txt

### Chave de relacionamento
IDAtracacao

### Técnica
Leitura do arquivo 2024Carga.txt com Pandas e chunksize.

### Parâmetros do teste
- Tamanho do bloco: 50.000 linhas
- Quantidade de blocos: 2
- Máximo de registros nacionais analisados: 100.000

### Cuidados metodológicos
- Preservar Porto Atracação e Instalação Portuária.
- Não interpretar o teste como movimentação anual.
- Não modificar os arquivos brutos.
- Validar o processamento completo antes de publicar indicadores.



## Aula 7 — Processamento completo e gravação incremental

### Objetivo
Processar o arquivo nacional 2024Carga.txt em blocos,
selecionando cargas vinculadas ao Complexo Portuário
de Santos.

### Fontes
- 2024Carga.txt
- 2024Atracacao.txt

### Regra de seleção
Complexo Portuário = Santos
Relacionamento: IDAtracacao

### Estratégia
1. Ler a base em blocos de 50.000 registros.
2. Identificar as atracações pertencentes ao complexo.
3. Converter o peso bruto para formato numérico.
4. Verificar falhas de conversão.
5. Gravar os registros selecionados em arquivo CSV.
6. Validar quantidade de registros e peso gravado.

### Cuidados
- Preservar o arquivo original.
- Não excluir registros com peso inválido silenciosamente.
- Não assumir que a seleção representa o porto organizado.
- Não interpretar o resultado parcial como total anual.


In [33]:
from pathlib import Path
import pandas as pd

# Diretórios do projeto
pasta_saida = Path("/content/ed_antaq_piloto/data")
pasta_saida.mkdir(parents=True, exist_ok=True)

arquivo_origem = Path("/content/2024Carga.txt")

arquivo_temporario = (
    pasta_saida / "2024Carga_Santos_processando.csv"
)

arquivo_final = (
    pasta_saida / "2024Carga_ComplexoSantos.csv"
)

# Chaves das atracações selecionadas
ids_santos = set(
    df_santos_atracacoes["IDAtracacao"]
    .astype("string")
)

print("Arquivo de origem:", arquivo_origem.name)
print("IDs selecionados:", len(ids_santos))
print("Pasta de saída:", pasta_saida)


Arquivo de origem: 2024Carga.txt
IDs selecionados: 6292
Pasta de saída: /content/ed_antaq_piloto/data


In [34]:

# AULA 7 - PIPELINE DE EXTRAÇÃO E TRANSFORMAÇÃO

import os

tamanho_bloco = 50000

total_lidos = 0
total_selecionados = 0
total_falhas_peso = 0
peso_processado = 0.0
blocos_processados = 0

# Eliminar apenas um arquivo temporário de execução anterior
arquivo_temporario.unlink(missing_ok=True)

# Garantir leitura textual dos identificadores e do peso
tipos = {
    "IDCarga": "string",
    "IDAtracacao": "string",
    "CDMercadoria": "string",
    "VLPesoCargaBruta": "string"
}

leitor = pd.read_csv(
    arquivo_origem,
    sep=";",
    encoding="utf-8-sig",
    dtype=tipos,
    chunksize=tamanho_bloco
)

try:
    for numero, bloco in enumerate(leitor, start=1):

        # 1. Registrar o volume recebido
        total_lidos += len(bloco)
        blocos_processados += 1

        # 2. Selecionar cargas do Complexo de Santos
        filtro_santos = (
            bloco["IDAtracacao"].isin(ids_santos)
        )

        selecionado = bloco.loc[filtro_santos].copy()

        # 3. Preservar o campo original e criar medida numérica
        selecionado["PesoBrutoNumerico"] = pd.to_numeric(
            selecionado["VLPesoCargaBruta"]
            .str.replace(",", ".", regex=False),
            errors="coerce"
        )

        # 4. Verificar falhas de conversão
        falhas = (
            selecionado["VLPesoCargaBruta"].notna()
            & selecionado["PesoBrutoNumerico"].isna()
        )

        total_falhas_peso += int(falhas.sum())

        if falhas.any():
            exemplos = selecionado.loc[
                falhas, "VLPesoCargaBruta"
            ].head(5).tolist()

            raise ValueError(
                f"Falha no bloco {numero}: {exemplos}"
            )

        # 5. Atualizar indicadores de controle
        total_selecionados += len(selecionado)

        peso_processado += (
            selecionado["PesoBrutoNumerico"].sum()
        )

        # 6. Gravar o bloco em disco
        if not selecionado.empty:

            primeiro_bloco_gravado = (
                not arquivo_temporario.exists()
            )

            selecionado.to_csv(
                arquivo_temporario,
                sep=";",
                index=False,
                mode="a",
                header=primeiro_bloco_gravado,
                encoding="utf-8"
            )

        if numero % 5 == 0:
            print(
                f"Blocos: {numero} | "
                f"Lidos: {total_lidos:,} | "
                f"Santos: {total_selecionados:,}"
            )

    # 7. Publicar o arquivo somente após concluir o fluxo
    if not arquivo_temporario.exists():
        raise ValueError("Nenhuma carga foi selecionada.")

    os.replace(arquivo_temporario, arquivo_final)

except Exception:
    arquivo_temporario.unlink(missing_ok=True)
    raise

print("\nPROCESSAMENTO CONCLUÍDO")
print("Blocos processados:", blocos_processados)
print("Registros nacionais:", total_lidos)
print("Registros de Santos:", total_selecionados)
print("Falhas de peso:", total_falhas_peso)
print("Soma numérica do peso:", peso_processado)
print("Arquivo:", arquivo_final)


/tmp/ipykernel_3087/969264901.py:33: DtypeWarning: Columns (12,24) have mixed types. Specify dtype option on import or set low_memory=False.
  for numero, bloco in enumerate(leitor, start=1):
/tmp/ipykernel_3087/969264901.py:33: DtypeWarning: Columns (12,24) have mixed types. Specify dtype option on import or set low_memory=False.
  for numero, bloco in enumerate(leitor, start=1):
/tmp/ipykernel_3087/969264901.py:33: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  for numero, bloco in enumerate(leitor, start=1):


Blocos: 5 | Lidos: 250,000 | Santos: 103,941


/tmp/ipykernel_3087/969264901.py:33: DtypeWarning: Columns (12,24) have mixed types. Specify dtype option on import or set low_memory=False.
  for numero, bloco in enumerate(leitor, start=1):
/tmp/ipykernel_3087/969264901.py:33: DtypeWarning: Columns (12,24) have mixed types. Specify dtype option on import or set low_memory=False.
  for numero, bloco in enumerate(leitor, start=1):


Blocos: 10 | Lidos: 500,000 | Santos: 177,006


/tmp/ipykernel_3087/969264901.py:33: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  for numero, bloco in enumerate(leitor, start=1):
/tmp/ipykernel_3087/969264901.py:33: DtypeWarning: Columns (12,24) have mixed types. Specify dtype option on import or set low_memory=False.
  for numero, bloco in enumerate(leitor, start=1):


...

/tmp/ipykernel_3087/969264901.py:33: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  for numero, bloco in enumerate(leitor, start=1):


Blocos: 45 | Lidos: 2,250,000 | Santos: 736,701


/tmp/ipykernel_3087/969264901.py:33: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  for numero, bloco in enumerate(leitor, start=1):
/tmp/ipykernel_3087/969264901.py:33: DtypeWarning: Columns (12,24) have mixed types. Specify dtype option on import or set low_memory=False.
  for numero, bloco in enumerate(leitor, start=1):



PROCESSAMENTO CONCLUÍDO
Blocos processados: 49
Registros nacionais: 2439611
Registros de Santos: 760354
Falhas de peso: 0
Soma numérica do peso: 177073928.0790001
Arquivo: /content/ed_antaq_piloto/data/2024Carga_ComplexoSantos.csv


/tmp/ipykernel_3087/969264901.py:33: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  for numero, bloco in enumerate(leitor, start=1):


In [35]:

# AULA 7 - VALIDAÇÃO DO ARQUIVO DE SAÍDA

linhas_gravadas = 0
peso_gravado = 0.0

for bloco in pd.read_csv(
    arquivo_final,
    sep=";",
    encoding="utf-8",
    chunksize=50000,
    dtype={"IDAtracacao": "string"}
):

    linhas_gravadas += len(bloco)

    peso_gravado += (
        bloco["PesoBrutoNumerico"].sum()
    )

print("Registros processados:", total_selecionados)
print("Registros gravados:", linhas_gravadas)

print("\nPeso processado:", peso_processado)
print("Peso gravado:", peso_gravado)

# Validação da quantidade de registros
assert linhas_gravadas == total_selecionados, (
    "ERRO: quantidade gravada divergente!"
)

# Validação da soma de peso, com tolerância numérica
import math

assert math.isclose(
    peso_gravado,
    peso_processado,
    rel_tol=1e-10,
    abs_tol=0.01
), "ERRO: divergência no peso gravado!"

print("\nVALIDAÇÃO DE GRAVAÇÃO: OK")


/tmp/ipykernel_3087/68214058.py:6: DtypeWarning: Columns (12,24) have mixed types. Specify dtype option on import or set low_memory=False.
  for bloco in pd.read_csv(
/tmp/ipykernel_3087/68214058.py:6: DtypeWarning: Columns (12,24) have mixed types. Specify dtype option on import or set low_memory=False.
  for bloco in pd.read_csv(
/tmp/ipykernel_3087/68214058.py:6: DtypeWarning: Columns (12,24) have mixed types. Specify dtype option on import or set low_memory=False.
  for bloco in pd.read_csv(
/tmp/ipykernel_3087/68214058.py:6: DtypeWarning: Columns (12,24) have mixed types. Specify dtype option on import or set low_memory=False.
  for bloco in pd.read_csv(
/tmp/ipykernel_3087/68214058.py:6: DtypeWarning: Columns (12,24) have mixed types. Specify dtype option on import or set low_memory=False.
  for bloco in pd.read_csv(
/tmp/ipykernel_3087/68214058.py:6: DtypeWarning: Columns (12,24) have mixed types. Specify dtype option on import or set low_memory=False.
  for bloco in pd.read_csv

Registros processados: 760354
Registros gravados: 760354

Peso processado: 177073928.0790001
Peso gravado: 177073928.07899997

VALIDAÇÃO DE GRAVAÇÃO: OK


/tmp/ipykernel_3087/68214058.py:6: DtypeWarning: Columns (12,24) have mixed types. Specify dtype option on import or set low_memory=False.
  for bloco in pd.read_csv(



### Resultado do processamento completo

**Fonte:** ANTAQ - 2024Carga.txt

**Recorte:** Complexo Portuário de Santos

**Estratégia:** Leitura em blocos de 50.000 linhas

#### Indicadores de execução
- Total de blocos:
- Total nacional de registros lidos:
- Total de registros selecionados:
- Falhas na conversão de peso:
- Total de linhas gravadas:
- Soma numérica do peso processado:
- Soma numérica do peso gravado:

#### Regras aplicadas
- Relacionamento por IDAtracacao.
- Seleção pelo Complexo Portuário = Santos.
- Preservação do campo original de peso.
- Conversão numérica em campo adicional.
- Gravação incremental em arquivo CSV.

#### Limitações
- Unidade do peso a confirmar no dicionário oficial.
- Integridade e duplicidades do arquivo completo a validar.
- Arquivo final ainda não representa uma camada Silver
  plenamente validada.

#### Próxima etapa
Construção do banco SQL e verificação de chaves.



## Aula 8 — Fundamentos de SQL e carga em SQLite

### Objetivo
Criar um banco de dados relacional para armazenar
e consultar os registros do Complexo Portuário de Santos.

### Fonte
2024Carga_ComplexoSantos.csv

### Tecnologias
- Python
- Pandas
- SQLite
- SQL

### Etapas
1. Criar conexão com SQLite.
2. Definir a estrutura inicial da tabela.
3. Carregar registros em blocos.
4. Validar a carga.
5. Executar consultas SQL.

### Regras de qualidade
- Não alterar o arquivo CSV original.
- Preservar identificadores como texto.
- Verificar duplicidade de IDCarga.
- Reconciliar registros e pesos.
- Não descartar registros silenciosamente.

### Limitação
A estrutura inicial será didática.
O modelo relacional definitivo dependerá
das validações completas das chaves e do
dicionário oficial da ANTAQ.


In [36]:

import sqlite3
from pathlib import Path

# Caminho do banco de dados
arquivo_banco = Path(
    "/content/ed_antaq_piloto/antaq_santos_2024.db"
)

arquivo_banco.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Criar conexão
conexao = sqlite3.connect(arquivo_banco)

print("Banco conectado com sucesso!")
print("Arquivo:", arquivo_banco)


Banco conectado com sucesso!
Arquivo: /content/ed_antaq_piloto/antaq_santos_2024.db


In [37]:

# AULA 8 - CRIAÇÃO DA TABELA

sql_criar_tabela = """
CREATE TABLE IF NOT EXISTS carga_santos (
    IDCarga TEXT,
    IDAtracacao TEXT,
    CDMercadoria TEXT,
    TipoOperacao TEXT,
    NaturezaCarga TEXT,
    Sentido TEXT,
    TEU INTEGER,
    QTCarga INTEGER,
    PesoBruto REAL
);
"""

conexao.execute(sql_criar_tabela)
conexao.commit()

print("Tabela carga_santos criada!")


Tabela carga_santos criada!


In [38]:

import pandas as pd

arquivo_csv = (
    "/content/ed_antaq_piloto/data/"
    "2024Carga_ComplexoSantos.csv"
)

colunas_origem = [
    "IDCarga",
    "IDAtracacao",
    "CDMercadoria",
    "Tipo Operação da Carga",
    "Natureza da Carga",
    "Sentido",
    "TEU",
    "QTCarga",
    "PesoBrutoNumerico"
]

renomear = {
    "Tipo Operação da Carga": "TipoOperacao",
    "Natureza da Carga": "NaturezaCarga",
    "PesoBrutoNumerico": "PesoBruto"
}

total_carregado = 0

try:
    # Uma transação para a carga inteira
    with conexao:

        # Esvaziar apenas a tabela de destino
        conexao.execute("DELETE FROM carga_santos")

        for bloco in pd.read_csv(
            arquivo_csv,
            sep=";",
            encoding="utf-8",
            usecols=colunas_origem,
            dtype={
                "IDCarga": "string",
                "IDAtracacao": "string",
                "CDMercadoria": "string"
            },
            chunksize=50000
        ):

            bloco = bloco.rename(columns=renomear)

            # Inserir os registros no banco
            bloco.to_sql(
                "carga_santos",
                conexao,
                if_exists="append",
                index=False
            )

            total_carregado += len(bloco)

    print("Registros carregados:", total_carregado)

except Exception as erro:
    print("Erro na carga:", erro)
    raise


/tmp/ipykernel_3087/330170470.py:35: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  for bloco in pd.read_csv(
/tmp/ipykernel_3087/330170470.py:35: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  for bloco in pd.read_csv(
/tmp/ipykernel_3087/330170470.py:35: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  for bloco in pd.read_csv(
/tmp/ipykernel_3087/330170470.py:35: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  for bloco in pd.read_csv(
/tmp/ipykernel_3087/330170470.py:35: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  for bloco in pd.read_csv(
/tmp/ipykernel_3087/330170470.py:35: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  for bloco in pd.read_csv(
/tmp

Registros carregados: 760354


In [40]:

sql = """
SELECT
    COUNT(*) AS TotalRegistros,
    SUM(PesoBruto) AS PesoTotal,
    COUNT(DISTINCT IDAtracacao) AS AtracacoesDistintas
FROM carga_santos;
"""

resultado = pd.read_sql_query(sql, conexao)

display(resultado)



,TotalRegistros,PesoTotal,AtracacoesDistintas
0,760354,1.770739e+08,5916


In [44]:

# AULA 8 - VERIFICAR DUPLICIDADES NO SQLITE

# 1. Armazenar a consulta SQL como texto
sql_duplicidades = """
SELECT
    IDCarga,
    COUNT(*) AS Quantidade
FROM carga_santos
GROUP BY IDCarga
HAVING COUNT(*) > 1
ORDER BY Quantidade DESC
LIMIT 20;
"""

# 2. Executar a consulta no banco SQLite
df_duplicados = pd.read_sql_query(
    sql_duplicidades,
    conexao
)

# 3. Exibir o resultado
display(df_duplicados)


,IDCarga,Quantidade



### Resultado da Aula 8

**Banco:** an taq_santos_2024.db
**Tabela:** carga_santos

#### Indicadores
- Registros carregados:
- Peso bruto total:
- Atracações distintas:
- IDs de carga duplicados:

#### Conceitos aprendidos
- Banco de dados relacional
- SQLite
- CREATE TABLE
- INSERT / carga com to_sql
- SELECT
- COUNT
- SUM
- GROUP BY
- HAVING

#### Limitações
- Tabela simplificada para aprendizagem.
- Sem chave primária formal nesta etapa.
- Sem chave estrangeira formal.
- Unidade de peso ainda sujeita à validação
  documental.
- Arquivo SQLite armazenado temporariamente
  no ambiente Colab.

#### Próxima etapa
Criar uma tabela de atracações no banco,
definir relacionamento e realizar JOIN com SQL.



## Aula 9 — Modelagem relacional

### Objetivo
Relacionar as tabelas carga_santos e atracacoes
diretamente no banco SQLite.

### Estrutura
Tabela principal: carga_santos
Tabela complementar: atracacoes
Chave: IDAtracacao
Cardinalidade esperada: N:1

### Validações
- Unicidade de IDAtracacao na tabela atracacoes.
- Integridade referencial entre carga e atracacao.
- Preservação da quantidade de registros de carga.

### Resultados
- Atracações carregadas:
- Cargas carregadas:
- Cargas com correspondência:
- Cargas sem correspondência:

### Próxima etapa
Identificação da data de referência e construção
da tabela mensal de movimentação.


In [46]:

# AULA 9 - CRIAR A TABELA DE ATRACAÇÕES

conexao.execute("""
CREATE TABLE IF NOT EXISTS atracacoes (
    IDAtracacao TEXT PRIMARY KEY,
    PortoAtracacao TEXT,
    Instalacao TEXT,
    ComplexoPortuario TEXT
);
""")

conexao.commit()

print("Tabela atracacoes criada.")


Tabela atracacoes criada.


In [47]:

# Selecionar e renomear as colunas

colunas = {
    "IDAtracacao": "IDAtracacao",
    "Porto Atracação": "PortoAtracacao",
    "Apelido Instalação Portuária": "Instalacao",
    "Complexo Portuário": "ComplexoPortuario"
}

df_atracacoes_sql = (
    df_santos_atracacoes[list(colunas)]
    .rename(columns=colunas)
    .copy()
)

# Padronizar a chave como texto
df_atracacoes_sql["IDAtracacao"] = (
    df_atracacoes_sql["IDAtracacao"].astype("string")
)

# Conferir integridade antes da carga
assert df_atracacoes_sql["IDAtracacao"].notna().all()
assert df_atracacoes_sql["IDAtracacao"].is_unique

print("Atracações preparadas:", len(df_atracacoes_sql))

display(df_atracacoes_sql.head())


Atracações preparadas: 6292


,IDAtracacao,PortoAtracacao,Instalacao,ComplexoPortuario
10,1474170,Santos,NaN,Santos
13,1467104,Santos,NaN,Santos
34,1513682,Santos,NaN,Santos
43,1474174,Santos,NaN,Santos
44,1429397,Santos,NaN,Santos


In [48]:

# Carregar as atracações

# Recriar o conteúdo da tabela de destino
conexao.execute("DELETE FROM atracacoes")
conexao.commit()

df_atracacoes_sql.to_sql(
    "atracacoes",
    conexao,
    if_exists="append",
    index=False
)

print("Atracações carregadas no SQLite.")


Atracações carregadas no SQLite.


In [49]:

# AULA 9 - JOIN ENTRE CARGA E ATRACAÇÃO

sql_join = """
SELECT
    c.IDCarga,
    c.IDAtracacao,
    a.PortoAtracacao,
    a.Instalacao,
    a.ComplexoPortuario,
    c.CDMercadoria,
    c.NaturezaCarga,
    c.Sentido,
    c.PesoBruto
FROM carga_santos AS c
LEFT JOIN atracacoes AS a
    ON c.IDAtracacao = a.IDAtracacao
LIMIT 10;
"""

df_join = pd.read_sql_query(sql_join, conexao)

display(df_join)


,IDCarga,IDAtracacao,PortoAtracacao,Instalacao,ComplexoPortuario,CDMercadoria,NaturezaCarga,Sentido,PesoBruto
0,35450474,1405071,Terminal Integrador Portuário Luiz Antonio Mes...,None,Santos,2304,Granel Sólido,Embarcados,26300.0
1,35457571,1406996,Terminal Marítimo Dow,None,Santos,2910,Granel Líquido e Gasoso,Desembarcados,2443.0
2,35457572,1406996,Terminal Marítimo Dow,None,Santos,2905,Granel Líquido e Gasoso,Desembarcados,907.0
3,35457573,1406996,Terminal Marítimo Dow,None,Santos,2909,Granel Líquido e Gasoso,Desembarcados,150.0
4,35457574,1406996,Terminal Marítimo Dow,None,Santos,3907,Granel Líquido e Gasoso,Embarcados,350.0
5,35457738,1407042,Terminal Integrador Portuário Luiz Antonio Mes...,None,Santos,2503,Granel Sólido,Desembarcados,25480.0
6,35457739,1407042,Terminal Integrador Portuário Luiz Antonio Mes...,None,Santos,2503,Granel Sólido,Desembarcados,4000.0
7,35457740,1407042,Terminal Integrador Portuário Luiz Antonio Mes...,None,Santos,2503,Granel Sólido,Desembarcados,1000.0
8,35457741,1407042,Terminal Integrador Portuário Luiz Antonio Mes...,None,Santos,2503,Granel Sólido,Desembarcados,4000.0
9,35457742,1407042,Terminal Integrador Portuário Luiz Antonio Mes...,None,Santos,2503,Granel Sólido,Desembarcados,1000.0


In [50]:

# Verificar integridade referencial

sql_validacao = """
SELECT
    COUNT(*) AS TotalCargas,
    COUNT(a.IDAtracacao) AS CargasComAtracacao,
    COUNT(*) - COUNT(a.IDAtracacao) AS CargasSemAtracacao
FROM carga_santos AS c
LEFT JOIN atracacoes AS a
    ON c.IDAtracacao = a.IDAtracacao;
"""

resultado = pd.read_sql_query(
    sql_validacao,
    conexao
)

display(resultado)


,TotalCargas,CargasComAtracacao,CargasSemAtracacao
0,760354,760354,0



## Aula 10 — Dimensão temporal e agregação mensal

### Objetivo
Identificar a data de referência operacional da ANTAQ
e construir uma tabela analítica mensal em SQL.

### Etapas
1. Identificar os campos de data disponíveis.
2. Verificar formatos e valores ausentes.
3. Definir a regra de competência mensal.
4. Incorporar a data ao banco relacional.
5. Criar e validar a agregação mensal.

### Cuidados metodológicos
- Não assumir que atracação e movimentação
  da carga ocorrem necessariamente no mesmo mês.
- Preservar as datas originais.
- Documentar a regra temporal escolhida.
- Reconciliar o peso antes e depois da agregação.


In [51]:

# AULA 10 - INVESTIGAR CAMPOS DE DATA DA ANTAQ

palavras_data = [
    "data",
    "dt",
    "inicio",
    "fim",
    "chegada",
    "saida",
    "desatrac"
]

colunas_temporais = [
    coluna
    for coluna in df_atracacao.columns
    if any(
        palavra in coluna.lower()
        for palavra in palavras_data
    )
]

print("COLUNAS TEMPORAIS IDENTIFICADAS:")
print("-" * 45)

for coluna in colunas_temporais:
    print(coluna)


COLUNAS TEMPORAIS IDENTIFICADAS:
---------------------------------------------
CDTUP
Data Atracação
Data Chegada
Data Desatracação
Data Início Operação
Data Término Operação


In [52]:

# Visualizar os valores das colunas temporais

if colunas_temporais:
    display(
        df_atracacao[
            ["IDAtracacao"] + colunas_temporais
        ].head(10)
    )
else:
    print("Nenhuma coluna temporal encontrada pelo filtro.")


,IDAtracacao,CDTUP,Data Atracação,Data Chegada,Data Desatracação,Data Início Operação,Data Término Operação
0,1512136,BRNAT,02/11/2024 09:10:00,02/11/2024 09:00:00,06/11/2024 02:00:00,NaN,NaN
1,1466554,BRIBB,06/06/2024 14:30:00,06/06/2024 09:00:00,14/07/2024 19:10:00,NaN,NaN
2,1484269,BRIBB,11/08/2024 17:15:00,11/08/2024 17:00:00,14/08/2024 10:10:00,NaN,NaN
3,1433333,BRMCZ,15/03/2024 09:00:00,15/03/2024 08:00:00,18/03/2024 11:30:00,NaN,NaN
4,1489223,BRFOR,06/08/2024 08:49:00,03/08/2024 17:58:00,09/08/2024 11:00:00,NaN,NaN
5,1493075,BRNAT,29/08/2024 08:45:00,29/08/2024 08:40:00,04/09/2024 10:00:00,NaN,NaN
6,1443129,BRNAT,25/04/2024 09:20:00,25/04/2024 09:10:00,28/04/2024 15:30:00,NaN,NaN
7,1434366,BRNAT,20/03/2024 08:10:00,20/03/2024 08:00:00,23/03/2024 17:30:00,NaN,NaN
8,1441590,BRIBB,13/04/2024 14:50:00,13/04/2024 14:00:00,17/04/2024 08:23:00,NaN,NaN
9,1519093,BRSTM,22/05/2024 15:00:00,22/05/2024 14:06:00,22/05/2024 18:55:00,NaN,NaN


In [53]:

# AULA 10 - CONVERSÃO DA DATA DE ATRACAÇÃO

# Preservar o DataFrame original
df_atracacao_datas = df_atracacao.copy()

# Converter o campo de texto para datetime
df_atracacao_datas["DataAtracacaoConvertida"] = (
    pd.to_datetime(
        df_atracacao_datas["Data Atracação"],
        format="%d/%m/%Y %H:%M:%S",
        errors="coerce"
    )
)

# Comparar os valores originais e convertidos
display(
    df_atracacao_datas[
        [
            "IDAtracacao",
            "Data Atracação",
            "DataAtracacaoConvertida"
        ]
    ].head(10)
)


,IDAtracacao,Data Atracação,DataAtracacaoConvertida
0,1512136,02/11/2024 09:10:00,2024-11-02 09:10:00
1,1466554,06/06/2024 14:30:00,2024-06-06 14:30:00
2,1484269,11/08/2024 17:15:00,2024-08-11 17:15:00
3,1433333,15/03/2024 09:00:00,2024-03-15 09:00:00
4,1489223,06/08/2024 08:49:00,2024-08-06 08:49:00
5,1493075,29/08/2024 08:45:00,2024-08-29 08:45:00
6,1443129,25/04/2024 09:20:00,2024-04-25 09:20:00
7,1434366,20/03/2024 08:10:00,2024-03-20 08:10:00
8,1441590,13/04/2024 14:50:00,2024-04-13 14:50:00
9,1519093,22/05/2024 15:00:00,2024-05-22 15:00:00


In [54]:

# Verificar falhas de conversão

datas_originais = df_atracacao_datas[
    "Data Atracação"
]

datas_convertidas = df_atracacao_datas[
    "DataAtracacaoConvertida"
]

falhas = (
    datas_originais.notna()
    & datas_convertidas.isna()
)

print("Total de atracações:", len(df_atracacao_datas))

print(
    "Datas ausentes na origem:",
    datas_originais.isna().sum()
)

print(
    "Falhas de conversão:",
    falhas.sum()
)

print(
    "Datas válidas:",
    datas_convertidas.notna().sum()
)


Total de atracações: 106834
Datas ausentes na origem: 98
Falhas de conversão: 0
Datas válidas: 106736


In [55]:

# Criar campos derivados de tempo

df_atracacao_datas["Ano"] = (
    df_atracacao_datas["DataAtracacaoConvertida"]
    .dt.year
)

df_atracacao_datas["Mes"] = (
    df_atracacao_datas["DataAtracacaoConvertida"]
    .dt.month
)

# Criar competência no formato AAAA-MM
df_atracacao_datas["Competencia"] = (
    df_atracacao_datas["DataAtracacaoConvertida"]
    .dt.strftime("%Y-%m")
)

display(
    df_atracacao_datas[
        [
            "IDAtracacao",
            "DataAtracacaoConvertida",
            "Ano",
            "Mes",
            "Competencia"
        ]
    ].head(10)
)


,IDAtracacao,DataAtracacaoConvertida,Ano,Mes,Competencia
0,1512136,2024-11-02 09:10:00,2024.0,11.0,2024-11
1,1466554,2024-06-06 14:30:00,2024.0,6.0,2024-06
2,1484269,2024-08-11 17:15:00,2024.0,8.0,2024-08
3,1433333,2024-03-15 09:00:00,2024.0,3.0,2024-03
4,1489223,2024-08-06 08:49:00,2024.0,8.0,2024-08
5,1493075,2024-08-29 08:45:00,2024.0,8.0,2024-08
6,1443129,2024-04-25 09:20:00,2024.0,4.0,2024-04
7,1434366,2024-03-20 08:10:00,2024.0,3.0,2024-03
8,1441590,2024-04-13 14:50:00,2024.0,4.0,2024-04
9,1519093,2024-05-22 15:00:00,2024.0,5.0,2024-05


In [60]:

# ============================================================
# PROJETO ED-ANTAQ-01
# AULA 10 - TRATAMENTO E VALIDAÇÃO DA DIMENSÃO TEMPORAL
# Fonte: ANTAQ - 2024Atracacao.txt
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. PRESERVAR OS DADOS ORIGINAIS
# ------------------------------------------------------------

df_atracacao_datas = df_atracacao.copy()

print("1. BASE ORIGINAL")
print("Total de registros:", len(df_atracacao_datas))


# ------------------------------------------------------------
# 2. CONVERTER AS DATAS
# ------------------------------------------------------------

formato_data = "%d/%m/%Y %H:%M:%S"

df_atracacao_datas["DataAtracacaoConvertida"] = pd.to_datetime(
    df_atracacao_datas["Data Atracação"],
    format=formato_data,
    errors="coerce"
)

df_atracacao_datas["DataDesatracacaoConvertida"] = pd.to_datetime(
    df_atracacao_datas["Data Desatracação"],
    format=formato_data,
    errors="coerce"
)

print("\n2. DATAS CONVERTIDAS")

display(
    df_atracacao_datas[
        [
            "IDAtracacao",
            "Data Atracação",
            "DataAtracacaoConvertida",
            "Data Desatracação",
            "DataDesatracacaoConvertida"
        ]
    ].head(10)
)


# ------------------------------------------------------------
# 3. CRIAR ANO, MÊS E COMPETÊNCIA
# ------------------------------------------------------------

df_atracacao_datas["Ano"] = (
    df_atracacao_datas["DataAtracacaoConvertida"]
    .dt.year
    .astype("Int64")
)

df_atracacao_datas["Mes"] = (
    df_atracacao_datas["DataAtracacaoConvertida"]
    .dt.month
    .astype("Int64")
)

df_atracacao_datas["Competencia"] = (
    df_atracacao_datas["DataAtracacaoConvertida"]
    .dt.strftime("%Y-%m")
)

print("\n3. DIMENSÃO TEMPORAL")

display(
    df_atracacao_datas[
        [
            "IDAtracacao",
            "DataAtracacaoConvertida",
            "Ano",
            "Mes",
            "Competencia"
        ]
    ].head(10)
)


# ------------------------------------------------------------
# 4. VALIDAR DATAS DE ATRACAÇÃO
# ------------------------------------------------------------

total_atracacoes = len(df_atracacao_datas)

datas_ausentes = (
    df_atracacao_datas["Data Atracação"]
    .isna()
    .sum()
)

falhas_conversao = (
    df_atracacao_datas["Data Atracação"].notna()
    & df_atracacao_datas["DataAtracacaoConvertida"].isna()
).sum()

competencias_ausentes = (
    df_atracacao_datas["Competencia"]
    .isna()
    .sum()
)

print("\n4. QUALIDADE DA DIMENSÃO TEMPORAL")
print("Total de atracações:", total_atracacoes)
print("Datas ausentes na origem:", datas_ausentes)
print("Falhas de conversão:", falhas_conversao)
print("Competências ausentes:", competencias_ausentes)


# ------------------------------------------------------------
# 5. DISTRIBUIÇÃO POR COMPETÊNCIA
# ------------------------------------------------------------

resumo_competencias = (
    df_atracacao_datas
    .groupby("Competencia", dropna=False)
    .size()
    .reset_index(name="QtdAtracacoes")
    .sort_values("Competencia", na_position="last")
)

print("\n5. DISTRIBUIÇÃO DAS ATRACAÇÕES POR COMPETÊNCIA")

display(resumo_competencias)


# ------------------------------------------------------------
# 6. IDENTIFICAR ATRACAÇÕES QUE ATRAVESSAM MESES
# ------------------------------------------------------------

mes_atracacao = (
    df_atracacao_datas["DataAtracacaoConvertida"]
    .dt.to_period("M")
)

mes_desatracacao = (
    df_atracacao_datas["DataDesatracacaoConvertida"]
    .dt.to_period("M")
)

# Considerar somente registros com as duas datas válidas

datas_validas = (
    df_atracacao_datas["DataAtracacaoConvertida"].notna()
    & df_atracacao_datas["DataDesatracacaoConvertida"].notna()
)

atravessam_meses = (
    datas_validas
    & (mes_atracacao != mes_desatracacao)
)

total_atravessam = int(atravessam_meses.sum())
total_datas_validas = int(datas_validas.sum())

if total_datas_validas > 0:
    percentual_atravessam = (
        total_atravessam / total_datas_validas
    ) * 100
else:
    percentual_atravessam = 0.0

print("\n6. ATRACAÇÕES QUE ATRAVESSAM MESES")

print("Atracações com ambas as datas válidas:", total_datas_validas)
print("Atracações que atravessam meses:", total_atravessam)
print("Percentual:", round(percentual_atravessam, 2), "%")


# ------------------------------------------------------------
# 7. VERIFICAR COMPETÊNCIAS FORA DE 2024
# ------------------------------------------------------------

fora_2024 = (
    df_atracacao_datas["Ano"].notna()
    & df_atracacao_datas["Ano"].ne(2024)
)

print("\n7. VERIFICAÇÃO DO ANO DE REFERÊNCIA")

print(
    "Atracações com ano diferente de 2024:",
    int(fora_2024.sum())
)

if fora_2024.any():
    display(
        df_atracacao_datas.loc[
            fora_2024,
            [
                "IDAtracacao",
                "Data Atracação",
                "Ano",
                "Competencia"
            ]
        ].head(10)
    )


# ------------------------------------------------------------
# 8. VALIDAÇÃO FINAL
# ------------------------------------------------------------

assert len(df_atracacao_datas) == len(df_atracacao), (
    "Erro: a quantidade de registros foi alterada."
)

assert (
    df_atracacao_datas["Ano"].dtype.name == "Int64"
), "Erro no tipo da coluna Ano."

assert (
    df_atracacao_datas["Mes"].dtype.name == "Int64"
), "Erro no tipo da coluna Mes."

assert (
    resumo_competencias["QtdAtracacoes"].sum()
    == total_atracacoes
), "Erro: a distribuição não corresponde ao total."

print("\n" + "=" * 50)
print("AULA 10 - PROCESSAMENTO CONCLUÍDO")
print("Estrutura temporal criada e controles executados.")
print("Revise os indicadores antes de aprovar a qualidade.")
print("=" * 50)


1. BASE ORIGINAL
Total de registros: 106834

2. DATAS CONVERTIDAS


,IDAtracacao,Data Atracação,DataAtracacaoConvertida,Data Desatracação,DataDesatracacaoConvertida
0,1512136,02/11/2024 09:10:00,2024-11-02 09:10:00,06/11/2024 02:00:00,2024-11-06 02:00:00
1,1466554,06/06/2024 14:30:00,2024-06-06 14:30:00,14/07/2024 19:10:00,2024-07-14 19:10:00
2,1484269,11/08/2024 17:15:00,2024-08-11 17:15:00,14/08/2024 10:10:00,2024-08-14 10:10:00
3,1433333,15/03/2024 09:00:00,2024-03-15 09:00:00,18/03/2024 11:30:00,2024-03-18 11:30:00
4,1489223,06/08/2024 08:49:00,2024-08-06 08:49:00,09/08/2024 11:00:00,2024-08-09 11:00:00
5,1493075,29/08/2024 08:45:00,2024-08-29 08:45:00,04/09/2024 10:00:00,2024-09-04 10:00:00
6,1443129,25/04/2024 09:20:00,2024-04-25 09:20:00,28/04/2024 15:30:00,2024-04-28 15:30:00
7,1434366,20/03/2024 08:10:00,2024-03-20 08:10:00,23/03/2024 17:30:00,2024-03-23 17:30:00
8,1441590,13/04/2024 14:50:00,2024-04-13 14:50:00,17/04/2024 08:23:00,2024-04-17 08:23:00
9,1519093,22/05/2024 15:00:00,2024-05-22 15:00:00,22/05/2024 18:55:00,2024-05-22 18:55:00



3. DIMENSÃO TEMPORAL


,IDAtracacao,DataAtracacaoConvertida,Ano,Mes,Competencia
0,1512136,2024-11-02 09:10:00,2024,11,2024-11
1,1466554,2024-06-06 14:30:00,2024,6,2024-06
2,1484269,2024-08-11 17:15:00,2024,8,2024-08
3,1433333,2024-03-15 09:00:00,2024,3,2024-03
4,1489223,2024-08-06 08:49:00,2024,8,2024-08
5,1493075,2024-08-29 08:45:00,2024,8,2024-08
6,1443129,2024-04-25 09:20:00,2024,4,2024-04
7,1434366,2024-03-20 08:10:00,2024,3,2024-03
8,1441590,2024-04-13 14:50:00,2024,4,2024-04
9,1519093,2024-05-22 15:00:00,2024,5,2024-05



4. QUALIDADE DA DIMENSÃO TEMPORAL
Total de atracações: 106834
Datas ausentes na origem: 98
Falhas de conversão: 0
Competências ausentes: 98

5. DISTRIBUIÇÃO DAS ATRACAÇÕES POR COMPETÊNCIA


,Competencia,QtdAtracacoes
0,2018-03,1
1,2022-04,1
2,2022-10,1
3,2023-04,5
4,2023-07,1
5,2023-09,1
6,2023-10,1
7,2023-11,3
8,2023-12,288
9,2024-01,8196



6. ATRACAÇÕES QUE ATRAVESSAM MESES
Atracações com ambas as datas válidas: 106736
Atracações que atravessam meses: 4190
Percentual: 3.93 %

7. VERIFICAÇÃO DO ANO DE REFERÊNCIA
Atracações com ano diferente de 2024: 302


,IDAtracacao,Data Atracação,Ano,Competencia
240,1446725,20/10/2022 18:50:00,2022,2022-10
281,1405186,28/12/2023 17:10:00,2023,2023-12
577,1404799,31/12/2023 13:21:00,2023,2023-12
1401,1406197,22/12/2023 17:32:00,2023,2023-12
1833,1406482,31/12/2023 07:26:00,2023,2023-12
4782,1404800,31/12/2023 16:21:00,2023,2023-12
5218,1404985,14/12/2023 06:40:00,2023,2023-12
5294,1406716,30/12/2023 19:20:00,2023,2023-12
5533,1406761,31/12/2023 20:45:00,2023,2023-12
5582,1406792,31/12/2023 11:00:00,2023,2023-12



AULA 10 - PROCESSAMENTO CONCLUÍDO
Estrutura temporal criada e controles executados.
Revise os indicadores antes de aprovar a qualidade.



## Aula 10 — Dimensão Temporal

### Objetivo
Construir e validar a dimensão temporal da tabela de
atracações da ANTAQ para o ano de 2024.

### Transformações
- Conversão de Data Atracação para datetime.
- Conversão de Data Desatracação para datetime.
- Criação dos campos Ano, Mes e Competencia.
- Identificação de atracações que atravessam meses.
- Verificação de competências fora de 2024.

### Regra provisória
A competência representa o mês da atracação.

### Resultados
- Total de atracações:
- Datas ausentes:
- Falhas de conversão:
- Competências ausentes:
- Atracações que atravessam meses:
- Percentual:
- Atracações com ano diferente de 2024:

### Limitação
A competência de atracação ainda não foi reconciliada
com a metodologia oficial de movimentação da ANTAQ.

### Próxima etapa
Incorporar os campos temporais ao SQLite e construir
a primeira tabela Gold de movimentação mensal.


In [61]:

# ============================================================
# AULA 11 - ETAPA 1
# PREPARAÇÃO DA DIMENSÃO TEMPORAL NO SQLITE
# ============================================================

import pandas as pd
import sqlite3

# 1. Selecionar somente o Complexo Portuário de Santos
colunas = [
    "IDAtracacao",
    "Porto Atracação",
    "Apelido Instalação Portuária",
    "Complexo Portuário",
    "Competencia"
]

df_tempo_santos = df_atracacao_datas.loc[
    df_atracacao_datas["Complexo Portuário"] == "Santos",
    colunas
].copy()

# 2. Renomear para facilitar as consultas SQL
df_tempo_santos = df_tempo_santos.rename(
    columns={
        "Porto Atracação": "PortoAtracacao",
        "Apelido Instalação Portuária": "Instalacao",
        "Complexo Portuário": "ComplexoPortuario"
    }
)

# 3. Padronizar a chave de relacionamento
df_tempo_santos["IDAtracacao"] = (
    df_tempo_santos["IDAtracacao"].astype("string")
)

# 4. Validar a chave antes de carregar
assert df_tempo_santos["IDAtracacao"].notna().all()
assert df_tempo_santos["IDAtracacao"].is_unique

# 5. Criar tabela auxiliar no SQLite
# Esta operação substitui apenas a tabela auxiliar.
df_tempo_santos.to_sql(
    "atracacoes_temporal",
    conexao,
    if_exists="replace",
    index=False
)

# 6. Conferir o resultado
print("Atracações de Santos:", len(df_tempo_santos))

print(
    "Competências ausentes:",
    df_tempo_santos["Competencia"].isna().sum()
)

display(df_tempo_santos.head())


Atracações de Santos: 6292
Competências ausentes: 0


,IDAtracacao,PortoAtracacao,Instalacao,ComplexoPortuario,Competencia
10,1474170,Santos,NaN,Santos,2024-04
13,1467104,Santos,NaN,Santos,2024-06
34,1513682,Santos,NaN,Santos,2024-11
43,1474174,Santos,NaN,Santos,2024-07
44,1429397,Santos,NaN,Santos,2024-02


In [62]:

# ============================================================
# AULA 11 - ETAPA 1
# PREPARAÇÃO DA DIMENSÃO TEMPORAL NO SQLITE
# ============================================================

import pandas as pd
import sqlite3

# 1. Selecionar somente o Complexo Portuário de Santos
colunas = [
    "IDAtracacao",
    "Porto Atracação",
    "Apelido Instalação Portuária",
    "Complexo Portuário",
    "Competencia"
]

df_tempo_santos = df_atracacao_datas.loc[
    df_atracacao_datas["Complexo Portuário"] == "Santos",
    colunas
].copy()

# 2. Renomear para facilitar as consultas SQL
df_tempo_santos = df_tempo_santos.rename(
    columns={
        "Porto Atracação": "PortoAtracacao",
        "Apelido Instalação Portuária": "Instalacao",
        "Complexo Portuário": "ComplexoPortuario"
    }
)

# 3. Padronizar a chave de relacionamento
df_tempo_santos["IDAtracacao"] = (
    df_tempo_santos["IDAtracacao"].astype("string")
)

# 4. Validar a chave antes de carregar
assert df_tempo_santos["IDAtracacao"].notna().all()
assert df_tempo_santos["IDAtracacao"].is_unique

# 5. Criar tabela auxiliar no SQLite
# Esta operação substitui apenas a tabela auxiliar.
df_tempo_santos.to_sql(
    "atracacoes_temporal",
    conexao,
    if_exists="replace",
    index=False
)

# 6. Conferir o resultado
print("Atracações de Santos:", len(df_tempo_santos))

print(
    "Competências ausentes:",
    df_tempo_santos["Competencia"].isna().sum()
)

display(df_tempo_santos.head())


Atracações de Santos: 6292
Competências ausentes: 0


,IDAtracacao,PortoAtracacao,Instalacao,ComplexoPortuario,Competencia
10,1474170,Santos,NaN,Santos,2024-04
13,1467104,Santos,NaN,Santos,2024-06
34,1513682,Santos,NaN,Santos,2024-11
43,1474174,Santos,NaN,Santos,2024-07
44,1429397,Santos,NaN,Santos,2024-02


In [63]:

# ============================================================
# AULA 11 - ETAPA 2
# VALIDAÇÃO DA INTEGRIDADE REFERENCIAL
# ============================================================

sql_validacao = """
SELECT
    COUNT(*) AS TotalCargas,
    COUNT(a.IDAtracacao) AS ComAtracacao,
    COUNT(*) - COUNT(a.IDAtracacao) AS SemAtracacao
FROM carga_santos AS c
LEFT JOIN atracacoes_temporal AS a
    ON c.IDAtracacao = a.IDAtracacao;
"""

validacao = pd.read_sql_query(
    sql_validacao,
    conexao
)

display(validacao)

# Impedir avanço com referências não encontradas
assert validacao.loc[0, "SemAtracacao"] == 0, (
    "Existem cargas sem correspondência na tabela temporal."
)

print("Integridade referencial: OK")


,TotalCargas,ComAtracacao,SemAtracacao
0,760354,760354,0


Integridade referencial: OK


In [64]:

# ============================================================
# AULA 11 - ETAPA 3
# CRIAÇÃO DA TABELA GOLD DE MOVIMENTAÇÃO MENSAL
# ============================================================

sql_gold = """
CREATE TABLE gold_movimentacao_mensal AS
SELECT
    a.Competencia,
    a.PortoAtracacao,
    a.Instalacao,
    c.NaturezaCarga,
    c.Sentido,

    COUNT(*) AS QtdRegistrosCarga,

    SUM(c.PesoBruto) AS PesoBrutoTotal

FROM carga_santos AS c

LEFT JOIN atracacoes_temporal AS a
    ON c.IDAtracacao = a.IDAtracacao

GROUP BY
    a.Competencia,
    a.PortoAtracacao,
    a.Instalacao,
    c.NaturezaCarga,
    c.Sentido;
"""

# Reconstruir apenas a tabela analítica do piloto
with conexao:
    conexao.execute(
        "DROP TABLE IF EXISTS gold_movimentacao_mensal"
    )

    conexao.execute(sql_gold)

print("Tabela Gold criada com sucesso!")

resultado_gold = pd.read_sql_query(
    """
    SELECT *
    FROM gold_movimentacao_mensal
    ORDER BY Competencia, PortoAtracacao
    LIMIT 20;
    """,
    conexao
)

display(resultado_gold)


Tabela Gold criada com sucesso!


,Competencia,PortoAtracacao,Instalacao,NaturezaCarga,Sentido,QtdRegistrosCarga,PesoBrutoTotal
0,2023-12,Base Logística de Dutos,None,Carga Geral,Embarcados,1,295.951
1,2023-12,DP World Santos,Ex-Embraport,Carga Conteinerizada,Desembarcados,9,3159.110
2,2023-12,DP World Santos,Ex-Embraport,Carga Conteinerizada,Embarcados,14,10216.950
3,2023-12,DP World Santos,Ex-Embraport,Carga Conteinerizada,Não Informado,12,5799.120
4,2023-12,Santos,None,Carga Conteinerizada,Desembarcados,2058,39005.638
5,2023-12,Santos,None,Carga Conteinerizada,Embarcados,228,133355.927
6,2023-12,Santos,None,Carga Conteinerizada,Não Informado,328,18331.898
7,2023-12,Santos,None,Carga Geral,Embarcados,56,61803.875
8,2023-12,Santos,None,Granel Líquido e Gasoso,Desembarcados,15,43401.771
9,2023-12,Santos,None,Granel Líquido e Gasoso,Embarcados,2,32227.818


In [65]:

# ============================================================
# AULA 11 - ETAPA 4
# RECONCILIAÇÃO ENTRE TABELA DETALHADA E GOLD
# ============================================================

import math

sql_reconciliacao = """
SELECT
    'Detalhada' AS Tabela,
    COUNT(*) AS Registros,
    SUM(PesoBruto) AS PesoTotal
FROM carga_santos

UNION ALL

SELECT
    'Gold' AS Tabela,
    SUM(QtdRegistrosCarga) AS Registros,
    SUM(PesoBrutoTotal) AS PesoTotal
FROM gold_movimentacao_mensal;
"""

reconciliacao = pd.read_sql_query(
    sql_reconciliacao,
    conexao
)

display(reconciliacao)

linhas = reconciliacao.set_index("Tabela")

# Verificar preservação dos registros
assert (
    linhas.loc["Detalhada", "Registros"]
    == linhas.loc["Gold", "Registros"]
), "Divergência na quantidade de registros!"

# Verificar preservação da soma do peso
assert math.isclose(
    linhas.loc["Detalhada", "PesoTotal"],
    linhas.loc["Gold", "PesoTotal"],
    rel_tol=1e-10,
    abs_tol=0.01
), "Divergência no peso!"

print("Reconciliação aprovada!")


,Tabela,Registros,PesoTotal
0,Detalhada,760354,1.770739e+08
1,Gold,760354,1.770739e+08


Reconciliação aprovada!


In [66]:

# ============================================================
# AULA 11 - ETAPA 5
# CONSULTA ANALÍTICA MENSAL
# ============================================================

sql_mensal = """
SELECT
    Competencia,
    SUM(PesoBrutoTotal) AS PesoTotal
FROM gold_movimentacao_mensal
GROUP BY Competencia
ORDER BY Competencia;
"""

df_mensal = pd.read_sql_query(
    sql_mensal,
    conexao
)

display(df_mensal)


,Competencia,PesoTotal
0,2023-12,1.220451e+06
1,2024-01,1.166078e+07
2,2024-02,1.420672e+07
3,2024-03,1.570609e+07
4,2024-04,1.422675e+07
5,2024-05,1.585811e+07
6,2024-06,1.656172e+07
7,2024-07,1.595301e+07
8,2024-08,1.564622e+07
9,2024-09,1.585719e+07



## Aula 11 — Construção da camada Gold

### Objetivo
Construir uma tabela analítica mensal de cargas
associadas às atracações do Complexo Portuário
de Santos.

### Fonte
ANTAQ — Dados de 2024.

### Tabelas utilizadas
- carga_santos
- atracacoes_temporal

### Regra temporal
Competência definida pelo mês da atracação.

### Dimensões
- Competencia
- PortoAtracacao
- Instalacao
- NaturezaCarga
- Sentido

### Medidas
- QtdRegistrosCarga
- PesoBrutoTotal

### Validações
- Integridade referencial.
- Preservação da quantidade de registros.
- Reconciliação do peso total.

### Limitações
- Regra temporal provisória.
- Não equivale necessariamente à série oficial
  de movimentação mensal da ANTAQ.
- Unidade do peso sujeita à confirmação
  documental.
- Tabela Gold ainda não submetida aos testes
  completos de produção.

### Próxima etapa
Avaliar a consistência mensal, documentar
as regras e preparar a migração para VS Code.


In [67]:

# AULA 11 - VALIDAÇÃO DO RELACIONAMENTO SQL

sql_validacao = """
SELECT
    COUNT(*) AS TotalCargas,
    COUNT(a.IDAtracacao) AS ComAtracacao,
    COUNT(*) - COUNT(a.IDAtracacao) AS SemAtracacao
FROM carga_santos AS c
LEFT JOIN atracacoes_temporal AS a
    ON c.IDAtracacao = a.IDAtracacao;
"""

validacao = pd.read_sql_query(
    sql_validacao,
    conexao
)

display(validacao)

assert validacao.loc[0, "SemAtracacao"] == 0, (
    "Existem cargas sem atracação correspondente."
)

print("Validação de relacionamento concluída.")


,TotalCargas,ComAtracacao,SemAtracacao
0,760354,760354,0


Validação de relacionamento concluída.



### Validação da tabela atracacoes_temporal

- Atracações do Complexo de Santos: 6.292
- Competências ausentes: 0
- Chave de relacionamento: IDAtracacao
- Campo de competência: mês da Data Atracação

**Observação:**
O campo Instalacao pode estar ausente para
determinados registros, conforme a classificação
original da ANTAQ.

**Status:**
Tabela temporal criada.
Integridade referencial com carga_santos
ainda em validação.


In [68]:

# ============================================================
# AULA 11 - CRIACAO DA TABELA GOLD
# PROJETO ED-ANTAQ-01
# ============================================================

import pandas as pd

# 1. Confirmar que nao existem cargas sem atracacao
sql_controle = """
SELECT
    COUNT(*) AS TotalLinhasJoin,
    COUNT(a.IDAtracacao) AS ComAtracacao,
    COUNT(*) - COUNT(a.IDAtracacao) AS SemAtracacao
FROM carga_santos AS c
LEFT JOIN atracacoes_temporal AS a
    ON c.IDAtracacao = a.IDAtracacao;
"""

controle = pd.read_sql_query(sql_controle, conexao)

total_cargas = pd.read_sql_query(
    "SELECT COUNT(*) AS total FROM carga_santos",
    conexao
).loc[0, "total"]

assert controle.loc[0, "SemAtracacao"] == 0, (
    "Existem cargas sem atracacao correspondente."
)

assert controle.loc[0, "TotalLinhasJoin"] == total_cargas, (
    "O JOIN multiplicou a quantidade de cargas."
)

# 2. Criar a tabela Gold
sql_gold = """
CREATE TABLE gold_movimentacao_mensal AS
SELECT
    a.Competencia,
    a.PortoAtracacao,
    c.NaturezaCarga,
    c.Sentido,
    COUNT(*) AS QtdRegistrosCarga,
    SUM(c.PesoBruto) AS PesoBrutoTotal
FROM carga_santos AS c
INNER JOIN atracacoes_temporal AS a
    ON c.IDAtracacao = a.IDAtracacao
GROUP BY
    a.Competencia,
    a.PortoAtracacao,
    c.NaturezaCarga,
    c.Sentido;
"""

with conexao:
    conexao.execute(
        "DROP TABLE IF EXISTS gold_movimentacao_mensal"
    )
    conexao.execute(sql_gold)

# 3. Visualizar o resultado
df_gold = pd.read_sql_query(
    """
    SELECT *
    FROM gold_movimentacao_mensal
    ORDER BY Competencia, PortoAtracacao
    LIMIT 20
    """,
    conexao
)

print("Tabela Gold criada.")
display(df_gold)


Tabela Gold criada.


,Competencia,PortoAtracacao,NaturezaCarga,Sentido,QtdRegistrosCarga,PesoBrutoTotal
0,2023-12,Base Logística de Dutos,Carga Geral,Embarcados,1,295.951
1,2023-12,DP World Santos,Carga Conteinerizada,Desembarcados,9,3159.110
2,2023-12,DP World Santos,Carga Conteinerizada,Embarcados,14,10216.950
3,2023-12,DP World Santos,Carga Conteinerizada,Não Informado,12,5799.120
4,2023-12,Santos,Carga Conteinerizada,Desembarcados,2058,39005.638
5,2023-12,Santos,Carga Conteinerizada,Embarcados,228,133355.927
6,2023-12,Santos,Carga Conteinerizada,Não Informado,328,18331.898
7,2023-12,Santos,Carga Geral,Embarcados,56,61803.875
8,2023-12,Santos,Granel Líquido e Gasoso,Desembarcados,15,43401.771
9,2023-12,Santos,Granel Líquido e Gasoso,Embarcados,2,32227.818


In [69]:

# ============================================================
# AULA 11 - RECONCILIACAO DA TABELA GOLD
# ============================================================

import math

sql_reconciliacao = """
SELECT
    'Detalhada' AS Camada,
    COUNT(*) AS QtdRegistros,
    SUM(PesoBruto) AS PesoTotal
FROM carga_santos

UNION ALL

SELECT
    'Gold' AS Camada,
    SUM(QtdRegistrosCarga) AS QtdRegistros,
    SUM(PesoBrutoTotal) AS PesoTotal
FROM gold_movimentacao_mensal;
"""

reconciliacao = pd.read_sql_query(
    sql_reconciliacao,
    conexao
)

display(reconciliacao)

detalhada = reconciliacao.iloc[0]
gold = reconciliacao.iloc[1]

assert detalhada["QtdRegistros"] == gold["QtdRegistros"], (
    "Divergencia na quantidade de registros."
)

assert math.isclose(
    detalhada["PesoTotal"],
    gold["PesoTotal"],
    rel_tol=1e-10,
    abs_tol=0.01
), "Divergencia no peso bruto."

print("Reconciliacao concluida com sucesso.")


,Camada,QtdRegistros,PesoTotal
0,Detalhada,760354,1.770739e+08
1,Gold,760354,1.770739e+08


Reconciliacao concluida com sucesso.


In [70]:

# ============================================================
# AULA 11 - CONSULTA MENSAL
# ============================================================

sql_mensal = """
SELECT
    Competencia,
    SUM(QtdRegistrosCarga) AS QtdRegistros,
    ROUND(SUM(PesoBrutoTotal), 2) AS PesoTotal
FROM gold_movimentacao_mensal
GROUP BY Competencia
ORDER BY Competencia;
"""

df_mensal = pd.read_sql_query(
    sql_mensal,
    conexao
)

display(df_mensal)


,Competencia,QtdRegistros,PesoTotal
0,2023-12,2941,1220450.77
1,2024-01,55368,11660779.30
2,2024-02,60226,14206716.77
3,2024-03,62204,15706092.94
4,2024-04,61986,14226747.96
5,2024-05,65714,15858109.01
6,2024-06,64245,16561720.96
7,2024-07,71107,15953012.52
8,2024-08,65559,15646215.91
9,2024-09,66549,15857193.55



## Aula 11 — Tabela Gold

### Objetivo
Construir uma tabela analítica com o peso bruto das
cargas associadas às atracações do Complexo de Santos.

### Fontes
- carga_santos
- atracacoes_temporal

### Regra temporal
Competência definida pelo mês da atracação.

### Transformações
- Relacionamento por IDAtracacao.
- Agrupamento por competência, porto, natureza e sentido.
- Contagem dos registros.
- Soma do peso bruto.

### Validações
- Correspondência entre cargas e atracações.
- Preservação da quantidade de registros.
- Reconciliação do peso total.

### Limitações
- Competência mensal provisória.
- Indicador não reconciliado com o painel oficial da ANTAQ.
- Verificação de unidade de medida ainda pendente.

### Próxima etapa
Revisar a série mensal e preparar a migração para VS Code.


In [71]:

# ============================================================
# PROJETO ED-ANTAQ-01
# AULA 11 - VALIDAÇÃO FINAL DA CAMADA GOLD
# ============================================================

import pandas as pd
import math

# 1. Reconciliar a tabela detalhada com a Gold
sql_reconciliacao = """
SELECT
    'Detalhada' AS Camada,
    COUNT(*) AS Registros,
    SUM(PesoBruto) AS PesoTotal
FROM carga_santos

UNION ALL

SELECT
    'Gold' AS Camada,
    SUM(QtdRegistrosCarga) AS Registros,
    SUM(PesoBrutoTotal) AS PesoTotal
FROM gold_movimentacao_mensal;
"""

reconciliacao = pd.read_sql_query(
    sql_reconciliacao, conexao
)

print("1. RECONCILIACAO")
display(reconciliacao)

assert (
    reconciliacao.loc[0, "Registros"]
    == reconciliacao.loc[1, "Registros"]
), "Divergencia na quantidade de registros!"

assert math.isclose(
    reconciliacao.loc[0, "PesoTotal"],
    reconciliacao.loc[1, "PesoTotal"],
    rel_tol=1e-10,
    abs_tol=0.01
), "Divergencia no peso!"

# 2. Identificar competencias fora de 2024
sql_fora_periodo = """
SELECT
    Competencia,
    SUM(QtdRegistrosCarga) AS Registros,
    SUM(PesoBrutoTotal) AS PesoTotal
FROM gold_movimentacao_mensal
WHERE Competencia IS NULL
   OR Competencia < '2024-01'
   OR Competencia >= '2025-01'
GROUP BY Competencia
ORDER BY Competencia;
"""

print("\n2. COMPETENCIAS FORA DO PERIODO")
display(pd.read_sql_query(sql_fora_periodo, conexao))

# 3. Criar uma VIEW analitica para 2024
with conexao:
    conexao.execute("""
        DROP VIEW IF EXISTS vw_movimentacao_2024
    """)

    conexao.execute("""
        CREATE VIEW vw_movimentacao_2024 AS
        SELECT *
        FROM gold_movimentacao_mensal
        WHERE Competencia >= '2024-01'
          AND Competencia < '2025-01'
    """)

# 4. Conferir a visão criada
sql_resumo = """
SELECT
    COUNT(DISTINCT Competencia) AS Meses,
    SUM(QtdRegistrosCarga) AS Registros,
    ROUND(SUM(PesoBrutoTotal), 2) AS PesoTotal
FROM vw_movimentacao_2024;
"""

print("\n3. RESUMO DA VIEW 2024")
display(pd.read_sql_query(sql_resumo, conexao))

print("\nValidacoes concluidas.")


1. RECONCILIACAO


,Camada,Registros,PesoTotal
0,Detalhada,760354,1.770739e+08
1,Gold,760354,1.770739e+08



2. COMPETENCIAS FORA DO PERIODO


,Competencia,Registros,PesoTotal
0,2023-12,2941,1220450.77



3. RESUMO DA VIEW 2024


,Meses,Registros,PesoTotal
0,12,757413,1.758535e+08



Validacoes concluidas.



### Validação da tabela Gold — Complexo de Santos

**Fonte:** ANTAQ, arquivo 2024Carga.txt.

**Regra temporal provisória:** mês da atracação.

**Resultados da consulta:**
- Competências identificadas: 2023-12 a 2024-12
- Registros totais: 760.354
- Registros classificados em 2024: 757.413
- Registros classificados em 2023-12: 2.941
- Peso associado a 2024: 175.853.477,30
- Peso associado a 2023-12: 1.220.450,77

**Decisão técnica:**
Preservar todas as competências na tabela Gold.
Criar uma VIEW específica para o ano de 2024.

**Validações pendentes:**
- Confirmar reconciliação entre detalhada e Gold.
- Confirmar unidade de peso da fonte.
- Comparar série com o Estatístico Aquaviário.
- Confirmar regra oficial de competência mensal.
- Investigar registros de virada de ano.


In [72]:

# AULA 12 - BACKUP DO PROJETO

from pathlib import Path
import shutil
from google.colab import files

pasta_projeto = Path("/content/ed_antaq_piloto")
arquivo_banco = pasta_projeto / "antaq_santos_2024.db"

# Confirmar que os arquivos necessários existem
assert pasta_projeto.exists(), "Pasta do projeto não encontrada."
assert arquivo_banco.exists(), "Banco SQLite não encontrado."

# Concluir operações pendentes no banco
conexao.commit()

# Criar uma cópia consistente do banco
import sqlite3

copia_banco = pasta_projeto / "antaq_santos_2024_backup.db"

with sqlite3.connect(copia_banco) as destino:
    conexao.backup(destino)

# Compactar a pasta do projeto
arquivo_zip = shutil.make_archive(
    "/content/ED_ANTAQ_01_backup",
    "zip",
    root_dir="/content",
    base_dir="ed_antaq_piloto"
)

print("Backup criado:", arquivo_zip)

# Baixar para o computador
files.download(arquivo_zip)


Backup criado: /content/ED_ANTAQ_01_backup.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


## Encerramento da Aula 11 — Validação da Gold

### Resultado
- Registros reconciliados: 760.354
- Registros classificados em 2024: 757.413
- Registros classificados em dezembro de 2023: 2.941
- Meses disponíveis na visão de 2024: 12

### Validações aprovadas
- Contagem de registros entre Detalhada e Gold.
- Reconciliação do peso bruto.
- Identificação das competências fora de 2024.
- Criação da VIEW analítica de 2024.

### Pendências metodológicas
- Confirmar unidade de peso.
- Confirmar competência oficial da ANTAQ.
- Comparar os resultados com o Estatístico Aquaviário.
- Implementar controles adicionais de qualidade.

### Próxima fase
Migração do Google Colab para o VS Code.
Modularização do pipeline em Python.
